## In this notebook will train simple baseline models using the target dataset we just created

Use only information available at date T.
Do not use any future_* or target_* columns as input features.

### Setup and load target dataset

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start_path: Path) -> Path:
    """
    Walk upward from current notebook folder until we find the project root.
    Project root is identified by config.yaml or README.md.
    """
    start_path = start_path.resolve()

    for path in [start_path] + list(start_path.parents):
        if (path / "config.yaml").exists() or (path / "README.md").exists():
            return path

    raise FileNotFoundError("Project root not found. Expected config.yaml or README.md.")


PROJECT_ROOT = find_project_root(Path.cwd())

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
TARGET_DIR = INTERIM_DIR / "targets"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "baseline_modeling"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_INPUT_PARQUET = TARGET_DIR / "stock_features_with_targets_v1.parquet"
TARGET_INPUT_CSV = TARGET_DIR / "stock_features_with_targets_v1.csv"

BASELINE_REPORT_PATH = REPORT_DIR / "baseline_model_results_v1.csv"
BASELINE_FEATURE_IMPORTANCE_PATH = REPORT_DIR / "baseline_feature_importance_v1.csv"

print("Project root:", PROJECT_ROOT)
print("Target parquet:", TARGET_INPUT_PARQUET)
print("Model dir:", MODEL_DIR)
print("Report dir:", REPORT_DIR)

if TARGET_INPUT_PARQUET.exists():
    data = pd.read_parquet(TARGET_INPUT_PARQUET)
    print("Loaded from parquet")
else:
    data = pd.read_csv(TARGET_INPUT_CSV, parse_dates=["date"])
    print("Loaded from CSV")

data["date"] = pd.to_datetime(data["date"])

data = (
    data
    .sort_values(["yf_ticker", "date"])
    .reset_index(drop=True)
)

print("Data shape:", data.shape)
print("Date range:", data["date"].min(), "to", data["date"].max())
print("Unique stocks:", data["yf_ticker"].nunique())
# to display all the columns & rows in the DataFrame without truncation
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

display(data.head())

Project root: E:\Projects\marketguard-india
Target parquet: E:\Projects\marketguard-india\data\interim\targets\stock_features_with_targets_v1.parquet
Model dir: E:\Projects\marketguard-india\models
Report dir: E:\Projects\marketguard-india\reports\baseline_modeling
Loaded from parquet
Data shape: (357370, 200)
Date range: 2010-01-04 00:00:00 to 2026-07-14 00:00:00
Unique stocks: 100


,asset_type,symbol,yf_ticker,company_name,industry,date,adj_close,close,dividends,high,low,open,repaired?,stock_splits,volume,source_file,history_days_available,prev_close,prev_adj_close,return_1d,return_5d,return_20d,return_60d,return_120d,log_return_1d,raw_return_1d,daily_volatility_20d,daily_volatility_60d,annualized_volatility_20d,annualized_volatility_60d,ma_20,adj_close_to_ma_20,ma_50,adj_close_to_ma_50,ma_100,adj_close_to_ma_100,ma_200,adj_close_to_ma_200,rolling_high_60d,rolling_high_252d,drawdown_from_60d_high,drawdown_from_252d_high,rsi_14,above_ma_20,above_ma_50,above_ma_200,ma_20_above_ma_50,ma_50_above_ma_200,true_range,atr_14,atr_14_pct,intraday_range_pct,open_gap_pct,close_position_in_day_range,volume_ma_20,volume_ma_60,volume_ratio_20d,volume_ratio_60d,turnover,turnover_ma_20,turnover_ratio_20d,large_raw_move_flag,stock_acceptance_status,stock_history_rows,stock_history_start_date,stock_history_end_date,is_research_universe,is_limited_history,large_move_reason,review_status,large_move_review_flag,manual_review_large_move_flag,keep_large_move_flag,nifty50_close,nifty50_return_1d,nifty50_return_5d,nifty50_return_20d,nifty50_return_60d,nifty50_daily_volatility_20d,nifty50_daily_volatility_60d,nifty50_annualized_volatility_20d,nifty50_annualized_volatility_60d,nifty50_close_to_ma_20,nifty50_close_to_ma_50,nifty50_close_to_ma_200,nifty100_close,nifty100_return_1d,nifty100_return_5d,nifty100_return_20d,nifty100_return_60d,nifty100_daily_volatility_20d,nifty100_daily_volatility_60d,nifty100_annualized_volatility_20d,nifty100_annualized_volatility_60d,nifty100_close_to_ma_20,nifty100_close_to_ma_50,nifty100_close_to_ma_200,indiavix_close,indiavix_return_1d,indiavix_return_5d,indiavix_return_20d,indiavix_return_60d,indiavix_daily_volatility_20d,indiavix_daily_volatility_60d,indiavix_annualized_volatility_20d,indiavix_annualized_volatility_60d,indiavix_close_to_ma_20,indiavix_close_to_ma_50,indiavix_close_to_ma_200,sector_proxy_ticker,sector_proxy_name,sector_close,sector_return_1d,sector_return_5d,sector_return_20d,sector_return_60d,sector_daily_volatility_20d,sector_daily_volatility_60d,sector_annualized_volatility_20d,sector_annualized_volatility_60d,sector_close_to_ma_20,sector_close_to_ma_50,sector_close_to_ma_200,relative_return_20d_vs_nifty50,relative_return_60d_vs_nifty50,relative_return_20d_vs_nifty100,relative_return_60d_vs_nifty100,relative_return_20d_vs_sector,relative_return_60d_vs_sector,volatility_ratio_vs_nifty50_20d,volatility_ratio_vs_nifty100_20d,volatility_ratio_vs_sector_20d,nifty50_uptrend_flag,nifty100_uptrend_flag,vix_above_20_flag,vix_above_25_flag,vix_20d_change,base_features_ready,market_features_ready,sector_features_ready,model_ready_v1,model_ready_v1_clean,future_adj_close_5d,future_return_5d,target_available_5d,future_adj_close_20d,future_return_20d,target_available_20d,future_adj_close_60d,future_return_60d,target_available_60d,future_min_adj_close_5d,future_max_adj_close_5d,future_min_return_5d,future_max_return_5d,path_target_available_5d,future_min_adj_close_20d,future_max_adj_close_20d,future_min_return_20d,future_max_return_20d,path_target_available_20d,future_min_adj_close_60d,future_max_adj_close_60d,future_min_return_60d,future_max_return_60d,path_target_available_60d,target_ready_v1_20d,target_ready_v1_clean_20d,target_positive_return_20d,target_downside_5pct_20d,target_big_downside_10pct_20d,target_upside_5pct_20d,target_return_bucket_20d,target_scenario_20d,target_scenario_id_20d,future_upside_downside_ratio_20d,target_ready_v1_5d,target_ready_v1_clean_5d,target_ready_v1_60d,target_ready_v1_clean_60d,future_nifty50_close_5d,future_nifty50_return_5d,future_excess_return_5d_vs_nifty50,future_sector_close_5d,future_sector_return_5d,future_excess_return_5d_vs_sector,future_nifty50_close_20d,future_nifty50_return_20d,future_excess_return_20d_vs_nifty50,future_sector_close_20d,future_sector_return_20d,future_excess_return_20d_vs_sector,future_nifty50_close_60d,future_nifty50_return_60

### Define modelling target, safe features, and time split

Now we define a time-based train/validation/test split and choose safe input features.

We will not use random split because stock market data is time-series data.

In [2]:
# Main baseline modelling target for v1
# 1 if the stock outperformed Nifty 50 over the next 20 trading days
# 0 otherwise
TARGET_COL = "target_outperform_nifty50_20d"
READY_COL = "target_ready_v1_20d"

# Time-based split:
# Train on older history, validate on 2024, test on newer unseen period.
TRAIN_END_DATE = pd.Timestamp("2023-12-31")
VALID_START_DATE = pd.Timestamp("2024-01-01")
VALID_END_DATE = pd.Timestamp("2024-12-31")
TEST_START_DATE = pd.Timestamp("2025-01-01")

model_data = data[data[READY_COL] == 1].copy()

train_mask = model_data["date"] <= TRAIN_END_DATE

valid_mask = (
    (model_data["date"] >= VALID_START_DATE)
    & (model_data["date"] <= VALID_END_DATE)
)

test_mask = model_data["date"] >= TEST_START_DATE

# Columns that must never be used as model inputs.
# This avoids leakage from future targets or labels.
leakage_prefixes = (
    "future_",
    "target_",
)

manual_exclude_cols = [
    # Identity / text / metadata
    "date",
    "symbol",
    "yf_ticker",
    "company_name",
    "industry",
    "source_file",
    "sector_proxy_ticker",
    "sector_proxy_name",
    "stock_acceptance_status",
    "large_move_reason",
    "review_status",

    # Raw price/volume levels.
    # We prefer normalized returns, ratios, volatility, drawdown, RSI, regime features.
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume",
    "prev_close",
    "prev_adj_close",
]

candidate_feature_cols = []

for col in model_data.columns:
    if col in manual_exclude_cols:
        continue

    if col.startswith(leakage_prefixes):
        continue

    if pd.api.types.is_numeric_dtype(model_data[col]):
        candidate_feature_cols.append(col)

# Remove columns with all missing values in model data
feature_cols = [
    col for col in candidate_feature_cols
    if model_data[col].notna().sum() > 0
]

print("Model data shape:", model_data.shape)
print("Target column:", TARGET_COL)
print("Ready column:", READY_COL)

print("\nSplit rows:")
print("Train:", train_mask.sum())
print("Valid:", valid_mask.sum())
print("Test:", test_mask.sum())
print("Total:", train_mask.sum() + valid_mask.sum() + test_mask.sum())

print("\nNumber of selected feature columns:", len(feature_cols))

print("\nTarget distribution by split:")
for split_name, mask in [
    ("train", train_mask),
    ("valid", valid_mask),
    ("test", test_mask),
]:
    print(f"\n{split_name}:")
    display(
        model_data.loc[mask, TARGET_COL]
        .value_counts(normalize=False)
        .to_frame("count")
        .assign(
            pct=lambda x: (x["count"] / x["count"].sum() * 100).round(2)
        )
    )

print("\n selected features:")
display(pd.DataFrame({"feature": feature_cols}).head(150))

print("\nLeakage check - selected columns starting with future_ or target_:")
display(
    pd.DataFrame(
        {
            "bad_feature": [
                col for col in feature_cols
                if col.startswith(leakage_prefixes)
            ]
        }
    )
)

Model data shape: (307453, 200)
Target column: target_outperform_nifty50_20d
Ready column: target_ready_v1_20d

Split rows:
Train: 253538
Valid: 21780
Test: 32135
Total: 307453

Number of selected feature columns: 123

Target distribution by split:

train:


,count,pct
target_outperform_nifty50_20d,,
1,130046,51.29
0,123492,48.71



valid:


,count,pct
target_outperform_nifty50_20d,,
1,10989,50.45
0,10791,49.55



test:


,count,pct
target_outperform_nifty50_20d,,
1,16623,51.73
0,15512,48.27



 selected features:


,feature
0,dividends
1,repaired?
2,stock_splits
3,history_days_available
4,return_1d
5,return_5d
6,return_20d
7,return_60d
8,return_120d
9,log_return_1d



Leakage check - selected columns starting with future_ or target_:


,bad_feature


### if you observe all the features , we dont need some of the features such as pipeline/control flags, these are not real market signals.

#### also we will avoid raw scale columns like: 

ma_20, ma_50, ma_100, ma_200

rolling_high_60d, rolling_high_252d

volume_ma_20, volume_ma_60

turnover

nifty50_close

nifty100_close

sector_close

Instead, use normalized features like returns, volatility, ratios, drawdowns, RSI, VIX, market regime, and relative strength.

In [3]:
# Clean v1 baseline feature set.
# We intentionally exclude:
# - future_* and target_* columns
# - readiness / availability flags
# - raw absolute price levels
# - raw moving-average levels
# - raw volume / turnover levels
# - metadata / identity columns

stock_return_features = [
    "return_1d",
    "return_5d",
    "return_20d",
    "return_60d",
    "return_120d",
    "log_return_1d",
    "raw_return_1d",
]

stock_risk_features = [
    "annualized_volatility_20d",
    "annualized_volatility_60d",
    "atr_14_pct",
    "intraday_range_pct",
    "open_gap_pct",
    "close_position_in_day_range",
]

stock_trend_features = [
    "adj_close_to_ma_20",
    "adj_close_to_ma_50",
    "adj_close_to_ma_100",
    "adj_close_to_ma_200",
    "drawdown_from_60d_high",
    "drawdown_from_252d_high",
    "rsi_14",
    "above_ma_20",
    "above_ma_50",
    "above_ma_200",
    "ma_20_above_ma_50",
    "ma_50_above_ma_200",
]

stock_volume_features = [
    "volume_ratio_20d",
    "volume_ratio_60d",
    "turnover_ratio_20d",
]

market_features = [
    "nifty50_return_1d",
    "nifty50_return_5d",
    "nifty50_return_20d",
    "nifty50_return_60d",
    "nifty50_annualized_volatility_20d",
    "nifty50_annualized_volatility_60d",
    "nifty50_close_to_ma_20",
    "nifty50_close_to_ma_50",
    "nifty50_close_to_ma_200",

    "nifty100_return_1d",
    "nifty100_return_5d",
    "nifty100_return_20d",
    "nifty100_return_60d",
    "nifty100_annualized_volatility_20d",
    "nifty100_annualized_volatility_60d",
    "nifty100_close_to_ma_20",
    "nifty100_close_to_ma_50",
    "nifty100_close_to_ma_200",
]

vix_features = [
    "indiavix_close",
    "indiavix_return_1d",
    "indiavix_return_5d",
    "indiavix_return_20d",
    "indiavix_return_60d",
    "indiavix_annualized_volatility_20d",
    "indiavix_annualized_volatility_60d",
    "indiavix_close_to_ma_20",
    "indiavix_close_to_ma_50",
    "indiavix_close_to_ma_200",
]

sector_features = [
    "sector_return_1d",
    "sector_return_5d",
    "sector_return_20d",
    "sector_return_60d",
    "sector_annualized_volatility_20d",
    "sector_annualized_volatility_60d",
    "sector_close_to_ma_20",
    "sector_close_to_ma_50",
    "sector_close_to_ma_200",
]

relative_strength_features = [
    "relative_return_20d_vs_nifty50",
    "relative_return_60d_vs_nifty50",
    "relative_return_20d_vs_nifty100",
    "relative_return_60d_vs_nifty100",
    "relative_return_20d_vs_sector",
    "relative_return_60d_vs_sector",
    "volatility_ratio_vs_nifty50_20d",
    "volatility_ratio_vs_nifty100_20d",
    "volatility_ratio_vs_sector_20d",
]

regime_features = [
    "nifty50_uptrend_flag",
    "nifty100_uptrend_flag",
    "vix_above_20_flag",
    "vix_above_25_flag",
    "vix_20d_change",
]

feature_cols = (
    stock_return_features
    + stock_risk_features
    + stock_trend_features
    + stock_volume_features
    + market_features
    + vix_features
    + sector_features
    + relative_strength_features
    + regime_features
)

missing_features = [
    col for col in feature_cols
    if col not in model_data.columns
]

print("Missing clean feature columns:", missing_features)

if missing_features:
    raise ValueError(f"Missing clean feature columns: {missing_features}")

bad_leakage_features = [
    col for col in feature_cols
    if col.startswith(("future_", "target_"))
]

bad_control_flags = [
    col for col in feature_cols
    if col in [
        "base_features_ready",
        "market_features_ready",
        "sector_features_ready",
        "model_ready_v1",
        "model_ready_v1_clean",
        "path_target_available_5d",
        "path_target_available_20d",
        "path_target_available_60d",
    ]
]

print("Clean feature count:", len(feature_cols))
print("Leakage features:", bad_leakage_features)
print("Control flags included:", bad_control_flags)

display(pd.DataFrame({"feature": feature_cols}))

Missing clean feature columns: []
Clean feature count: 79
Leakage features: []
Control flags included: []


,feature
0,return_1d
1,return_5d
2,return_20d
3,return_60d
4,return_120d
5,log_return_1d
6,raw_return_1d
7,annualized_volatility_20d
8,annualized_volatility_60d
9,atr_14_pct


### Build X/y train, validation, test sets

In [4]:
# Build model matrices

X_train = model_data.loc[train_mask, feature_cols].copy()
y_train = model_data.loc[train_mask, TARGET_COL].astype(int).copy()

X_valid = model_data.loc[valid_mask, feature_cols].copy()
y_valid = model_data.loc[valid_mask, TARGET_COL].astype(int).copy()

X_test = model_data.loc[test_mask, feature_cols].copy()
y_test = model_data.loc[test_mask, TARGET_COL].astype(int).copy()

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

print("\nTarget mean by split:")
print("Train positive rate:", round(y_train.mean() * 100, 2), "%")
print("Valid positive rate:", round(y_valid.mean() * 100, 2), "%")
print("Test positive rate:", round(y_test.mean() * 100, 2), "%")

# Missing-value audit for selected model features
missing_summary = pd.DataFrame(
    {
        "feature": feature_cols,
        "train_missing_pct": X_train.isna().mean().values * 100,
        "valid_missing_pct": X_valid.isna().mean().values * 100,
        "test_missing_pct": X_test.isna().mean().values * 100,
    }
)

missing_summary["max_missing_pct"] = missing_summary[
    ["train_missing_pct", "valid_missing_pct", "test_missing_pct"]
].max(axis=1)

missing_summary = missing_summary.sort_values(
    "max_missing_pct",
    ascending=False
).reset_index(drop=True)

print("\nFeatures with missing values:")
display(missing_summary[missing_summary["max_missing_pct"] > 0])

# Infinite-value audit
inf_summary = []

for split_name, X in [
    ("train", X_train),
    ("valid", X_valid),
    ("test", X_test),
]:
    for col in feature_cols:
        inf_count = np.isinf(pd.to_numeric(X[col], errors="coerce")).sum()

        if inf_count > 0:
            inf_summary.append(
                {
                    "split": split_name,
                    "feature": col,
                    "inf_count": inf_count,
                }
            )

inf_summary = pd.DataFrame(inf_summary)

print("\nInfinite value summary:")
display(inf_summary)

# Quick feature preview
display(X_train.head())
display(y_train.head())

X_train: (253538, 79) y_train: (253538,)
X_valid: (21780, 79) y_valid: (21780,)
X_test: (32135, 79) y_test: (32135,)

Target mean by split:
Train positive rate: 51.29 %
Valid positive rate: 50.45 %
Test positive rate: 51.73 %

Features with missing values:


,feature,train_missing_pct,valid_missing_pct,test_missing_pct,max_missing_pct
0,close_position_in_day_range,0.072179,0.036731,0.280068,0.280068



Infinite value summary:


""


,return_1d,return_5d,return_20d,return_60d,return_120d,log_return_1d,raw_return_1d,annualized_volatility_20d,annualized_volatility_60d,atr_14_pct,intraday_range_pct,open_gap_pct,close_position_in_day_range,adj_close_to_ma_20,adj_close_to_ma_50,adj_close_to_ma_100,adj_close_to_ma_200,drawdown_from_60d_high,drawdown_from_252d_high,rsi_14,above_ma_20,above_ma_50,above_ma_200,ma_20_above_ma_50,ma_50_above_ma_200,volume_ratio_20d,volume_ratio_60d,turnover_ratio_20d,nifty50_return_1d,nifty50_return_5d,nifty50_return_20d,nifty50_return_60d,nifty50_annualized_volatility_20d,nifty50_annualized_volatility_60d,nifty50_close_to_ma_20,nifty50_close_to_ma_50,nifty50_close_to_ma_200,nifty100_return_1d,nifty100_return_5d,nifty100_return_20d,nifty100_return_60d,nifty100_annualized_volatility_20d,nifty100_annualized_volatility_60d,nifty100_close_to_ma_20,nifty100_close_to_ma_50,nifty100_close_to_ma_200,indiavix_close,indiavix_return_1d,indiavix_return_5d,indiavix_return_20d,indiavix_return_60d,indiavix_annualized_volatility_20d,indiavix_annualized_volatility_60d,indiavix_close_to_ma_20,indiavix_close_to_ma_50,indiavix_close_to_ma_200,sector_return_1d,sector_return_5d,sector_return_20d,sector_return_60d,sector_annualized_volatility_20d,sector_annualized_volatility_60d,sector_close_to_ma_20,sector_close_to_ma_50,sector_close_to_ma_200,relative_return_20d_vs_nifty50,relative_return_60d_vs_nifty50,relative_return_20d_vs_nifty100,relative_return_60d_vs_nifty100,relative_return_20d_vs_sector,relative_return_60d_vs_sector,volatility_ratio_vs_nifty50_20d,volatility_ratio_vs_nifty100_20d,volatility_ratio_vs_sector_20d,nifty50_uptrend_flag,nifty100_uptrend_flag,vix_above_20_flag,vix_above_25_flag,vix_20d_change
251,-0.002650,0.015310,0.007409,-0.134983,-0.070004,-0.002654,-0.002650,0.274947,0.306878,0.025811,0.019900,0.000986,0.499998,0.027121,-0.013184,-0.027098,-0.024350,-0.127097,-0.147111,53.124928,1,0,0,0,0,1.481954,1.385218,1.508242,-0.001827,0.025075,0.025717,0.004256,0.141964,0.173696,0.026827,0.021686,0.102009,-0.002254,0.024969,0.018980,-0.008093,0.149044,0.174225,0.024951,0.013669,0.090854,17.030001,0.002354,-0.060673,-0.120805,-0.241425,0.726352,0.844409,-0.121463,-0.155819,-0.188265,-0.002254,0.024969,0.018980,-0.008093,0.149044,0.174225,0.024951,0.013669,0.090854,-0.018307,-0.139240,-0.011571,-0.126890,-0.011571,-0.126890,1.936738,1.844739,1.844739,1,1,0,0,-0.120805
252,-0.015759,-0.006178,-0.008591,-0.140065,-0.084923,-0.015885,-0.015759,0.280960,0.307733,0.026332,0.028695,0.000185,0.190371,0.011377,-0.026227,-0.042586,-0.039513,-0.140854,-0.160552,48.771730,1,0,0,0,0,0.981313,0.938784,0.983218,-0.010828,0.003209,0.017276,-0.003875,0.147844,0.175052,0.014834,0.010578,0.089222,-0.010947,0.003316,0.012109,-0.015947,0.153984,0.175547,0.013117,0.002824,0.078090,17.879999,0.049912,0.043783,-0.070686,-0.164486,0.752375,0.846357,-0.074367,-0.110395,-0.147327,-0.010947,0.003316,0.012109,-0.015947,0.153984,0.175547,0.013117,0.002824,0.078090,-0.025867,-0.136190,-0.020699,-0.124118,-0.020699,-0.124118,1.900377,1.824601,1.824601,1,1,0,0,-0.070686
253,0.003705,0.023302,0.008708,-0.133181,-0.084164,0.003698,0.003705,0.276826,0.307952,0.026033,0.023209,0.004395,0.584907,0.014679,-0.019888,-0.039231,-0.035775,-0.137670,-0.157442,49.796815,1,0,0,0,0,1.190590,1.173165,1.196830,-0.005189,-0.008784,0.024485,-0.014277,0.141581,0.175013,0.008351,0.005526,0.082772,-0.006818,-0.010181,0.018550,-0.028734,0.148047,0.175536,0.005288,-0.003521,0.069982,18.200001,0.017897,0.069330,-0.109153,-0.141509,0.717817,0.847134,-0.052330,-0.091617,-0.132081,-0.006818,-0.010181,0.018550,-0.028734,0.148047,0.175536,0.005288,-0.003521,0.069982,-0.015776,-0.118905,-0.009842,-0.104448,-0.009842,-0.104448,1.955246,1.869848,1.869848,1,1,0,0,-0.109153
254,-0.022834,-0.018659,0.027835,-0.138729,-0.083710,-0.023099,-0.022834,0.246445,0.309471,0.027067,0.028809,-0.000188,0.195555,-0.009819,-0.039444,-0.061236,-0.057566,-0.157361,-0.176681,43.939474,0,0,0,0,0,0.899754,0.904014,0.883429,-0

251    0
252    0
253    0
254    0
255    0
Name: target_outperform_nifty50_20d, dtype: int64

### Train dummy baseline and logistic regression

1. Dummy baseline → predicts majority class
2. Logistic regression → simple interpretable ML baseline

so , our target for Logistic regression is atleast beat baseline dummy model

### Model imports

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

### Model evaluation Function

In [6]:
def evaluate_classifier(model, X, y, split_name: str, model_name: str) -> dict:
    """
    Evaluate classifier on one split.
    Uses predicted probabilities when available.
    """
    y_pred = model.predict(X)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X)[:, 1]
    else:
        y_proba = y_pred

    return {
        "model": model_name,
        "split": split_name,
        "rows": len(y),
        "positive_rate_actual": y.mean(),
        "positive_rate_predicted": y_pred.mean(),
        "accuracy": accuracy_score(y, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_proba),
        "average_precision": average_precision_score(y, y_proba),
    }

### Dummy model

In [7]:
baseline_models = {}

# Model 1: Dummy baseline
# This gives us the minimum benchmark to beat.
dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train, y_train)

baseline_models["dummy_most_frequent"] = dummy_model


### Logistic regression

In [8]:
# Model 2: Logistic regression baseline
# Median imputation handles the small missing values.
# Scaling helps logistic regression learn stable coefficients.
logistic_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                solver="lbfgs",
                class_weight=None,
                random_state=42,
            ),
        ),
    ]
)

logistic_model.fit(X_train, y_train)

baseline_models["logistic_regression"] = logistic_model

# Evaluate models

### dummy model results

In [9]:
baseline_results = []

for model_name, model in baseline_models.items():
    for split_name, X, y in [
        ("train", X_train, y_train),
        ("valid", X_valid, y_valid),
        ("test", X_test, y_test),
    ]:
        baseline_results.append(
            evaluate_classifier(
                model=model,
                X=X,
                y=y,
                split_name=split_name,
                model_name=model_name,
            )
        )

baseline_results_df = pd.DataFrame(baseline_results)

display(
    baseline_results_df.sort_values(
        ["model", "split"]
    ).reset_index(drop=True)
)

print("\nValidation confusion matrices:")

for model_name, model in baseline_models.items():
    y_valid_pred = model.predict(X_valid)

    cm = confusion_matrix(y_valid, y_valid_pred)

    print(f"\n{model_name}")
    display(
        pd.DataFrame(
            cm,
            index=["actual_0", "actual_1"],
            columns=["pred_0", "pred_1"],
        )
    )



,model,split,rows,positive_rate_actual,positive_rate_predicted,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,dummy_most_frequent,test,32135,0.517286,1.000000,0.517286,0.500000,0.517286,1.000000,0.681857,0.500000,0.517286
1,dummy_most_frequent,train,253538,0.512925,1.000000,0.512925,0.500000,0.512925,1.000000,0.678057,0.500000,0.512925
2,dummy_most_frequent,valid,21780,0.504545,1.000000,0.504545,0.500000,0.504545,1.000000,0.670695,0.500000,0.504545
3,logistic_regression,test,32135,0.517286,0.757430,0.508791,0.499891,0.517214,0.757324,0.614652,0.500357,0.521656
4,logistic_regression,train,253538,0.512925,0.607881,0.543347,0.540585,0.546285,0.647417,0.592567,0.557243,0.559900
5,logistic_regression,valid,21780,0.504545,0.367723,0.490450,0.491652,0.493195,0.359450,0.415833,0.489352,0.503695



Validation confusion matrices:

dummy_most_frequent


,pred_0,pred_1
actual_0,0,10791
actual_1,0,10989



logistic_regression


,pred_0,pred_1
actual_0,6732,4059
actual_1,7039,3950


For validation, logistic regression is worse than dummy:

Dummy valid accuracy:      50.45%

Logistic valid accuracy:   49.05%

Dummy valid ROC AUC:       0.5000

Logistic valid ROC AUC:    0.4894

For test, logistic is basically random:

Logistic test ROC AUC: 0.5004

So, The Current logistic model is not useful.

### Now we know a simple linear model cannot capture the signal.

Possible reasons:

1. Outperforming Nifty is a hard target.
2. Logistic regression is too simple and linear.
3. Market behavior changes by year/regime.
4. Some useful relationships are nonlinear.
5. Maybe downside-risk target or scenario target is easier than outperform target.

Next, lets Try sinple a simple tree-based model. It can capture nonlinear rules like:

high volatility + weak sector + below MA200 = bad risk

strong relative strength + low VIX = better chance

## Train Random Forest baseline

In [10]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=8,
                min_samples_leaf=100,
                class_weight=None,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)


In [11]:

random_forest_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](79,)","['return_1d','return_5d','return_20d',...,'vix_above_20_flag', 'vix_above_25_flag','vix_20d_change']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,79
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_valu

In [12]:

baseline_models["random_forest"] = random_forest_model

rf_results = []

for split_name, X, y in [
    ("train", X_train, y_train),
    ("valid", X_valid, y_valid),
    ("test", X_test, y_test),
]:
    rf_results.append(
        evaluate_classifier(
            model=random_forest_model,
            X=X,
            y=y,
            split_name=split_name,
            model_name="random_forest",
        )
    )

rf_results_df = pd.DataFrame(rf_results)

display(rf_results_df)

print("\nValidation confusion matrix:")

y_valid_pred = random_forest_model.predict(X_valid)

display(
    pd.DataFrame(
        confusion_matrix(y_valid, y_valid_pred),
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
)

print("\nTest confusion matrix:")

y_test_pred = random_forest_model.predict(X_test)

display(
    pd.DataFrame(
        confusion_matrix(y_test, y_test_pred),
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
)

,model,split,rows,positive_rate_actual,positive_rate_predicted,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,random_forest,train,253538,0.512925,0.653894,0.609080,0.605172,0.593291,0.756348,0.664970,0.664625,0.671240
1,random_forest,valid,21780,0.504545,0.785491,0.516575,0.513981,0.513444,0.799345,0.625262,0.530184,0.523772
2,random_forest,test,32135,0.517286,0.725128,0.519309,0.511540,0.525234,0.736269,0.613100,0.519949,0.537898



Validation confusion matrix:


,pred_0,pred_1
actual_0,2467,8324
actual_1,2205,8784



Test confusion matrix:


,pred_0,pred_1
actual_0,4449,11063
actual_1,4384,12239


This is slightly better than dummy, but still weak.

Main result:

Random Forest valid ROC AUC: 0.5302
Random Forest test ROC AUC:  0.5199

So it is learning some signal, but not strong enough yet.

The problem is the default threshold 0.50. It predicts too many stocks as 1:

Actual positive rate valid:    50.45%
Predicted positive rate valid: 78.55%

So before judging it fully, tune the probability threshold on validation data.

### Threshold tuning for Random Forest

In [13]:
def evaluate_at_threshold(y_true, y_proba, threshold: float) -> dict:
    """
    Evaluate binary classifier at a custom probability threshold.
    """
    y_pred = (y_proba >= threshold).astype(int)

    return {
        "threshold": threshold,
        "positive_rate_predicted": y_pred.mean(),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


# Get predicted probabilities
rf_valid_proba = random_forest_model.predict_proba(X_valid)[:, 1]
rf_test_proba = random_forest_model.predict_proba(X_test)[:, 1]

# Try many thresholds
threshold_results = []

for threshold in np.arange(0.30, 0.71, 0.01):
    result = evaluate_at_threshold(
        y_true=y_valid,
        y_proba=rf_valid_proba,
        threshold=threshold,
    )
    threshold_results.append(result)

threshold_results_df = pd.DataFrame(threshold_results)

display(
    threshold_results_df
    .sort_values("balanced_accuracy", ascending=False)
    .head(20)
)

best_threshold = (
    threshold_results_df
    .sort_values("balanced_accuracy", ascending=False)
    .iloc[0]["threshold"]
)

print("Best validation threshold:", round(best_threshold, 2))

# Evaluate validation and test using best validation threshold
valid_threshold_metrics = evaluate_at_threshold(
    y_true=y_valid,
    y_proba=rf_valid_proba,
    threshold=best_threshold,
)

test_threshold_metrics = evaluate_at_threshold(
    y_true=y_test,
    y_proba=rf_test_proba,
    threshold=best_threshold,
)

threshold_final_df = pd.DataFrame(
    [
        {"split": "valid", **valid_threshold_metrics},
        {"split": "test", **test_threshold_metrics},
    ]
)

display(threshold_final_df)

print("\nValidation confusion matrix at tuned threshold:")

y_valid_pred_tuned = (rf_valid_proba >= best_threshold).astype(int)

display(
    pd.DataFrame(
        confusion_matrix(y_valid, y_valid_pred_tuned),
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
)

print("\nTest confusion matrix at tuned threshold:")

y_test_pred_tuned = (rf_test_proba >= best_threshold).astype(int)

display(
    pd.DataFrame(
        confusion_matrix(y_test, y_test_pred_tuned),
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
)

,threshold,positive_rate_predicted,accuracy,balanced_accuracy,precision,recall,f1
22,0.52,0.277961,0.520707,0.522727,0.545425,0.300482,0.387490
21,0.51,0.509780,0.522039,0.521951,0.526074,0.531532,0.528789
20,0.50,0.785491,0.516575,0.513981,0.513444,0.799345,0.625262
23,0.53,0.139945,0.505923,0.509197,0.537402,0.149058,0.233383
19,0.49,0.939348,0.510560,0.506567,0.508040,0.945855,0.661028
18,0.48,0.986915,0.507897,0.503471,0.506304,0.990354,0.670053
24,0.54,0.064876,0.497521,0.501476,0.515924,0.066339,0.117562
17,0.47,0.998072,0.504821,0.500293,0.504692,0.998362,0.670456
27,0.57,0.004729,0.495684,0.500187,0.524272,0.004914,0.009737
8,0.38,1.000000,0.504545,0.500000,0.504545,1.000000,0.670695


Best validation threshold: 0.52


,split,threshold,positive_rate_predicted,accuracy,balanced_accuracy,precision,recall,f1
0,valid,0.52,0.277961,0.520707,0.522727,0.545425,0.300482,0.387490
1,test,0.52,0.432363,0.511654,0.514009,0.533468,0.445888,0.485762



Validation confusion matrix at tuned threshold:


,pred_0,pred_1
actual_0,8039,2752
actual_1,7687,3302



Test confusion matrix at tuned threshold:


,pred_0,pred_1
actual_0,9030,6482
actual_1,9211,7412


This confirms the Random Forest is only slightly useful, not strong.

Best threshold result:

Validation balanced accuracy: 52.27%
Test balanced accuracy:       51.40%

So yes, it is better than random, but not enough to trust as a standalone model.

The more important question for this target is not only 0/1, but:

Do the top-ranked stocks actually outperform more often?

Because in the final dashboard, we will rank stocks by probability, not just classify everything as yes/no.

### Random Forest probability decile analysis

In [14]:
def probability_decile_analysis(eval_df: pd.DataFrame, proba: np.ndarray, split_name: str) -> pd.DataFrame:
    """
    Check whether higher model probability groups have better actual outcomes.

    Decile 10 = highest predicted probability group.
    Decile 1 = lowest predicted probability group.
    """
    temp = eval_df.copy()
    temp["predicted_probability"] = proba

    # Rank first to avoid qcut issues when many probabilities are similar.
    temp["probability_rank"] = temp["predicted_probability"].rank(method="first")

    temp["probability_decile"] = pd.qcut(
        temp["probability_rank"],
        q=10,
        labels=False,
    ) + 1

    summary = (
        temp
        .groupby("probability_decile")
        .agg(
            rows=("target_outperform_nifty50_20d", "size"),
            avg_predicted_probability=("predicted_probability", "mean"),
            actual_outperform_rate=("target_outperform_nifty50_20d", "mean"),
            avg_future_return_20d=("future_return_20d", "mean"),
            avg_nifty50_future_return_20d=("future_nifty50_return_20d", "mean"),
            avg_excess_return_vs_nifty50=("future_excess_return_20d_vs_nifty50", "mean"),
            downside_5pct_rate=("target_downside_5pct_20d", "mean"),
            big_downside_10pct_rate=("target_big_downside_10pct_20d", "mean"),
        )
        .reset_index()
        .sort_values("probability_decile", ascending=False)
    )

    summary["split"] = split_name

    return summary


valid_eval_df = model_data.loc[
    valid_mask,
    [
        "date",
        "symbol",
        "yf_ticker",
        "future_return_20d",
        "future_nifty50_return_20d",
        "future_excess_return_20d_vs_nifty50",
        "target_outperform_nifty50_20d",
        "target_downside_5pct_20d",
        "target_big_downside_10pct_20d",
    ]
].copy()

test_eval_df = model_data.loc[
    test_mask,
    [
        "date",
        "symbol",
        "yf_ticker",
        "future_return_20d",
        "future_nifty50_return_20d",
        "future_excess_return_20d_vs_nifty50",
        "target_outperform_nifty50_20d",
        "target_downside_5pct_20d",
        "target_big_downside_10pct_20d",
    ]
].copy()

valid_decile_summary = probability_decile_analysis(
    eval_df=valid_eval_df,
    proba=rf_valid_proba,
    split_name="valid",
)

test_decile_summary = probability_decile_analysis(
    eval_df=test_eval_df,
    proba=rf_test_proba,
    split_name="test",
)

print("Validation decile summary:")
display(valid_decile_summary)

print("\nTest decile summary:")
display(test_decile_summary)

print("\nTop decile vs bottom decile:")

top_bottom_summary = pd.concat(
    [
        valid_decile_summary[
            valid_decile_summary["probability_decile"].isin([1, 10])
        ],
        test_decile_summary[
            test_decile_summary["probability_decile"].isin([1, 10])
        ],
    ],
    ignore_index=True,
)

display(top_bottom_summary)

Validation decile summary:


,probability_decile,rows,avg_predicted_probability,actual_outperform_rate,avg_future_return_20d,avg_nifty50_future_return_20d,avg_excess_return_vs_nifty50,downside_5pct_rate,big_downside_10pct_rate,split
9,10,2178,0.546770,0.535354,0.028163,0.012675,0.015488,0.369605,0.139118,valid
8,9,2178,0.529195,0.552342,0.023852,0.008097,0.015756,0.380165,0.107897,valid
7,8,2178,0.521665,0.543159,0.018511,0.005165,0.013346,0.367309,0.109275,valid
6,7,2178,0.516469,0.498163,0.010921,0.003784,0.007137,0.394399,0.117539,valid
5,6,2178,0.512289,0.507346,0.011163,0.003457,0.007707,0.391185,0.132231,valid
4,5,2178,0.508484,0.491276,0.009987,0.005439,0.004547,0.406336,0.126263,valid
3,4,2178,0.504889,0.487144,0.011529,0.005301,0.006228,0.371442,0.105142,valid
2,3,2178,0.501334,0.493113,0.010147,0.004673,0.005475,0.39899,0.120753,valid
1,2,2178,0.496728,0.478421,0.011938,0.006750,0.005188,0.392562,0.131313,valid
0,1,2178,0.486905,0.459137,0.006942,0.013032,-0.006090,0.432048,0.16483,valid



Test decile summary:


,probability_decile,rows,avg_predicted_probability,actual_outperform_rate,avg_future_return_20d,avg_nifty50_future_return_20d,avg_excess_return_vs_nifty50,downside_5pct_rate,big_downside_10pct_rate,split
9,10,3214,0.565306,0.549471,0.028145,0.010446,0.017593,0.249222,0.05476,test
8,9,3213,0.545939,0.526299,0.013124,0.002820,0.010179,0.293495,0.089947,test
7,8,3214,0.534846,0.542937,0.006612,-0.001760,0.008138,0.351898,0.118544,test
6,7,3213,0.526172,0.515095,0.003958,-0.003275,0.007042,0.368192,0.116402,test
5,6,3213,0.518733,0.518207,0.004527,-0.001992,0.006196,0.355431,0.120448,test
4,5,3214,0.511612,0.523647,0.004547,-0.002152,0.006502,0.377722,0.126322,test
3,4,3213,0.504970,0.511983,0.005284,-0.001908,0.007076,0.377218,0.122316,test
2,3,3214,0.498245,0.503423,0.006448,0.001731,0.004605,0.378034,0.119477,test
1,2,3213,0.490602,0.490819,0.007888,0.004734,0.002787,0.382509,0.132586,test
0,1,3214,0.473973,0.490977,0.021502,0.015603,0.004906,0.391413,0.118855,test



Top decile vs bottom decile:


,probability_decile,rows,avg_predicted_probability,actual_outperform_rate,avg_future_return_20d,avg_nifty50_future_return_20d,avg_excess_return_vs_nifty50,downside_5pct_rate,big_downside_10pct_rate,split
0,10,2178,0.546770,0.535354,0.028163,0.012675,0.015488,0.369605,0.139118,valid
1,1,2178,0.486905,0.459137,0.006942,0.013032,-0.006090,0.432048,0.16483,valid
2,10,3214,0.565306,0.549471,0.028145,0.010446,0.017593,0.249222,0.05476,test
3,1,3214,0.473973,0.490977,0.021502,0.015603,0.004906,0.391413,0.118855,test


This result is actually more useful than the accuracy numbers showed.

The classifier is weak as a hard 0/1 predictor, but the ranking has some signal.

Validation

Top decile vs bottom decile:

Top decile actual outperform rate:     53.54%
Bottom decile actual outperform rate:  45.91%

Top decile excess return vs Nifty:     +1.55%
Bottom decile excess return vs Nifty:  -0.61%
Test
Top decile actual outperform rate:     54.95%
Bottom decile actual outperform rate:  49.10%

Top decile excess return vs Nifty:     +1.76%
Bottom decile excess return vs Nifty:  +0.49%

Also useful:

Test top decile big downside rate:     5.48%
Test bottom decile big downside rate:  11.89%

So the model is not great for saying:

This stock will definitely outperform: yes/no

But it is useful for:

Ranking stocks from stronger to weaker

### Random Forest feature importance
Now let’s check which features the Random Forest is using.

In [15]:
# Random Forest feature importance
# This tells us which features the model used most for splitting.
# Important note: tree feature importance is useful, but not perfect.
# Later we can use permutation importance or SHAP for stronger interpretation.

rf_estimator = random_forest_model.named_steps["model"]

rf_feature_importance = (
    pd.DataFrame(
        {
            "feature": feature_cols,
            "importance": rf_estimator.feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

rf_feature_importance["rank"] = rf_feature_importance.index + 1
rf_feature_importance["model"] = "random_forest"
rf_feature_importance["target"] = TARGET_COL

display(rf_feature_importance.head(40))

rf_feature_importance.to_csv(BASELINE_FEATURE_IMPORTANCE_PATH, index=False)

print("Saved:", BASELINE_FEATURE_IMPORTANCE_PATH)

print("\nTop 20 features:")
display(rf_feature_importance.head(20)[["rank", "feature", "importance"]])

,feature,importance,rank,model,target
0,sector_annualized_volatility_60d,0.044798,1,random_forest,target_outperform_nifty50_20d
1,sector_annualized_volatility_20d,0.038207,2,random_forest,target_outperform_nifty50_20d
2,indiavix_annualized_volatility_60d,0.036012,3,random_forest,target_outperform_nifty50_20d
3,nifty50_annualized_volatility_60d,0.034156,4,random_forest,target_outperform_nifty50_20d
4,nifty100_annualized_volatility_60d,0.029223,5,random_forest,target_outperform_nifty50_20d
5,annualized_volatility_60d,0.028070,6,random_forest,target_outperform_nifty50_20d
6,sector_close_to_ma_200,0.027244,7,random_forest,target_outperform_nifty50_20d
7,nifty100_close_to_ma_200,0.025563,8,random_forest,target_outperform_nifty50_20d
8,indiavix_close,0.024565,9,random_forest,target_outperform_nifty50_20d
9,indiavix_return_60d,0.024026,10,random_forest,target_outperform_nifty50_20d


Saved: E:\Projects\marketguard-india\reports\baseline_modeling\baseline_feature_importance_v1.csv

Top 20 features:


,rank,feature,importance
0,1,sector_annualized_volatility_60d,0.044798
1,2,sector_annualized_volatility_20d,0.038207
2,3,indiavix_annualized_volatility_60d,0.036012
3,4,nifty50_annualized_volatility_60d,0.034156
4,5,nifty100_annualized_volatility_60d,0.029223
5,6,annualized_volatility_60d,0.028070
6,7,sector_close_to_ma_200,0.027244
7,8,nifty100_close_to_ma_200,0.025563
8,9,indiavix_close,0.024565
9,10,indiavix_return_60d,0.024026


### Hyperparameter Tunning - Random search

import os
import time

from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterSampler


N_JOBS = -1
N_RANDOM_SEARCH_ITERATIONS = 20

RF_RANDOM_SEARCH_REPORT_PATH = REPORT_DIR / "rf_random_search_results_v1.csv"

print("Logical CPU cores available:", os.cpu_count())
print("Random Forest n_jobs:", N_JOBS)
print("Random search iterations:", N_RANDOM_SEARCH_ITERATIONS)


rf_param_space = {
    "n_estimators": [200, 300, 500],
    "max_depth": [4, 6, 8, 10, 12],
    "min_samples_leaf": [25, 50, 100, 200, 400],
    "min_samples_split": [50, 100, 200, 500],
    "max_features": ["sqrt", 0.30, 0.50, 0.70],
    "class_weight": [None, "balanced_subsample"],
    "bootstrap": [True],
}


rf_random_params = list(
    ParameterSampler(
        rf_param_space,
        n_iter=N_RANDOM_SEARCH_ITERATIONS,
        random_state=42,
    )
)

rf_random_search_results = []
rf_random_search_models = {}

best_valid_roc_auc_so_far = -1
best_rf_model_name_so_far = None


for i, params in enumerate(
    tqdm(rf_random_params, total=N_RANDOM_SEARCH_ITERATIONS, desc="RF random search"),
    start=1,
):
    model_name = f"rf_random_{i:02d}"

    print(f"\nTraining {model_name}/{N_RANDOM_SEARCH_ITERATIONS}")
    print(params)

    start_time = time.perf_counter()

    rf_model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    **params,
                    random_state=42,
                    n_jobs=N_JOBS,
                    verbose=0,
                ),
            ),
        ]
    )

    rf_model.fit(X_train, y_train)

    train_time_seconds = time.perf_counter() - start_time

    valid_metrics = evaluate_classifier(
        model=rf_model,
        X=X_valid,
        y=y_valid,
        split_name="valid",
        model_name=model_name,
    )

    result_row = {
        "model_name": model_name,
        "train_time_seconds": round(train_time_seconds, 2),
        **params,
        "valid_accuracy": valid_metrics["accuracy"],
        "valid_balanced_accuracy": valid_metrics["balanced_accuracy"],
        "valid_precision": valid_metrics["precision"],
        "valid_recall": valid_metrics["recall"],
        "valid_f1": valid_metrics["f1"],
        "valid_roc_auc": valid_metrics["roc_auc"],
        "valid_average_precision": valid_metrics["average_precision"],
        "valid_positive_rate_predicted": valid_metrics["positive_rate_predicted"],
    }

    rf_random_search_results.append(result_row)
    rf_random_search_models[model_name] = rf_model

    if valid_metrics["roc_auc"] > best_valid_roc_auc_so_far:
        best_valid_roc_auc_so_far = valid_metrics["roc_auc"]
        best_rf_model_name_so_far = model_name

    print(
        f"Finished {model_name} | "
        f"valid ROC AUC: {valid_metrics['roc_auc']:.4f} | "
        f"best so far: {best_rf_model_name_so_far} "
        f"({best_valid_roc_auc_so_far:.4f}) | "
        f"time: {train_time_seconds:.2f}s"
    )

    # Save progress after every model, so results are not lost if the notebook stops.
    pd.DataFrame(rf_random_search_results).to_csv(
        RF_RANDOM_SEARCH_REPORT_PATH,
        index=False,
    )


rf_random_search_results_df = (
    pd.DataFrame(rf_random_search_results)
    .sort_values("valid_roc_auc", ascending=False)
    .reset_index(drop=True)
)

display(rf_random_search_results_df)

best_rf_model_name = rf_random_search_results_df.iloc[0]["model_name"]
best_rf_model = rf_random_search_models[best_rf_model_name]

baseline_models["best_random_forest"] = best_rf_model

print("Best Random Forest model:", best_rf_model_name)

print("\nBest validation result:")
display(rf_random_search_results_df.head(1))


# Evaluate selected best model on train/valid/test
best_rf_results = []

for split_name, X, y in [
    ("train", X_train, y_train),
    ("valid", X_valid, y_valid),
    ("test", X_test, y_test),
]:
    best_rf_results.append(
        evaluate_classifier(
            model=best_rf_model,
            X=X,
            y=y,
            split_name=split_name,
            model_name="best_random_forest",
        )
    )

best_rf_results_df = pd.DataFrame(best_rf_results)

print("\nBest Random Forest train/valid/test results:")
display(best_rf_results_df)

print("\nValidation confusion matrix:")

y_valid_pred_best_rf = best_rf_model.predict(X_valid)

display(
    pd.DataFrame(
        confusion_matrix(y_valid, y_valid_pred_best_rf),
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
)

print("\nTest confusion matrix:")

y_test_pred_best_rf = best_rf_model.predict(X_test)

display(
    pd.DataFrame(
        confusion_matrix(y_test, y_test_pred_best_rf),
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
)

print("\nSaved random search results to:")
print(RF_RANDOM_SEARCH_REPORT_PATH)

Random search worked, but the result is mixed.

Previous RF valid ROC AUC: 0.5302
Tuned RF valid ROC AUC:    0.5404  improved

Previous RF test ROC AUC:  0.5199
Tuned RF test ROC AUC:     0.5179  slightly worse

So tuning helped on validation, but did not improve test performance. That means the Random Forest has some ranking signal, but it is still weak.

Also the tuned model is predicting too many 1s:

Actual positive rate test:     51.73%
Predicted positive rate test:  65.04%

So now we should tune the threshold for the best Random Forest and then run decile analysis again.

### Recovery Cell — Rebuild best Random Forest safely - if kernel died do not rerun random search just run this recover cell 

In [16]:
print("kernel is working")

kernel is working


In [17]:
from sklearn.ensemble import RandomForestClassifier

SAFE_N_JOBS = 15

best_rf_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                min_samples_split=200,
                min_samples_leaf=400,
                max_features=0.7,
                max_depth=10,
                class_weight=None,
                bootstrap=True,
                random_state=42,
                n_jobs=SAFE_N_JOBS,
            ),
        ),
    ]
)

best_rf_model.fit(X_train, y_train)



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](79,)","['return_1d','return_5d','return_20d',...,'vix_above_20_flag', 'vix_above_25_flag','vix_20d_change']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,79
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_valu

In [18]:
try:
    baseline_models
except NameError:
    baseline_models = {}

baseline_models["best_random_forest"] = best_rf_model

print("best_random_forest stored in baseline_models")
print(type(best_rf_model))

best_random_forest stored in baseline_models
<class 'sklearn.pipeline.Pipeline'>


### Threshold tuning for best Random Forest

In [19]:
def evaluate_at_threshold(y_true, y_proba, threshold: float) -> dict:
    """
    Evaluate binary classifier at a custom probability threshold.
    """
    y_pred = (y_proba >= threshold).astype(int)

    return {
        "threshold": threshold,
        "positive_rate_predicted": y_pred.mean(),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


best_rf_valid_proba = best_rf_model.predict_proba(X_valid)[:, 1]
best_rf_test_proba = best_rf_model.predict_proba(X_test)[:, 1]

best_rf_threshold_results = []

for threshold in np.arange(0.30, 0.80, 0.01):
    best_rf_threshold_results.append(
        evaluate_at_threshold(
            y_true=y_valid,
            y_proba=best_rf_valid_proba,
            threshold=threshold,
        )
    )

best_rf_threshold_results_df = pd.DataFrame(best_rf_threshold_results)

display(
    best_rf_threshold_results_df
    .sort_values("balanced_accuracy", ascending=False)
    .head(20)
)

best_rf_threshold = (
    best_rf_threshold_results_df
    .sort_values("balanced_accuracy", ascending=False)
    .iloc[0]["threshold"]
)

print("Best RF threshold from validation:", round(best_rf_threshold, 2))

best_rf_valid_threshold_metrics = evaluate_at_threshold(
    y_true=y_valid,
    y_proba=best_rf_valid_proba,
    threshold=best_rf_threshold,
)

best_rf_test_threshold_metrics = evaluate_at_threshold(
    y_true=y_test,
    y_proba=best_rf_test_proba,
    threshold=best_rf_threshold,
)

best_rf_threshold_final_df = pd.DataFrame(
    [
        {"model": "best_random_forest", "split": "valid", **best_rf_valid_threshold_metrics},
        {"model": "best_random_forest", "split": "test", **best_rf_test_threshold_metrics},
    ]
)

display(best_rf_threshold_final_df)

print("\nValidation confusion matrix at tuned threshold:")

best_rf_valid_pred_tuned = (best_rf_valid_proba >= best_rf_threshold).astype(int)

display(
    pd.DataFrame(
        confusion_matrix(y_valid, best_rf_valid_pred_tuned),
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
)

print("\nTest confusion matrix at tuned threshold:")

best_rf_test_pred_tuned = (best_rf_test_proba >= best_rf_threshold).astype(int)

display(
    pd.DataFrame(
        confusion_matrix(y_test, best_rf_test_pred_tuned),
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
)

,threshold,positive_rate_predicted,accuracy,balanced_accuracy,precision,recall,f1
23,0.53,0.292011,0.530624,0.532518,0.560220,0.324233,0.410744
22,0.52,0.407576,0.528834,0.529676,0.540949,0.436982,0.483439
21,0.51,0.542149,0.527870,0.527489,0.529895,0.569388,0.548932
24,0.54,0.192654,0.523646,0.526442,0.573165,0.218855,0.316760
20,0.50,0.681635,0.524931,0.523282,0.521622,0.704705,0.599497
25,0.55,0.121625,0.516253,0.519695,0.585504,0.141141,0.227453
19,0.49,0.801882,0.516621,0.513878,0.513198,0.815634,0.629999
26,0.56,0.075436,0.507438,0.511299,0.579428,0.086632,0.150728
18,0.48,0.888522,0.510882,0.507350,0.508681,0.895805,0.648891
17,0.47,0.943848,0.511111,0.507077,0.508294,0.950860,0.662461


Best RF threshold from validation: 0.53


,model,split,threshold,positive_rate_predicted,accuracy,balanced_accuracy,precision,recall,f1
0,best_random_forest,valid,0.53,0.292011,0.530624,0.532518,0.560220,0.324233,0.410744
1,best_random_forest,test,0.53,0.375541,0.503874,0.508187,0.528174,0.383445,0.444321



Validation confusion matrix at tuned threshold:


,pred_0,pred_1
actual_0,7994,2797
actual_1,7426,3563



Test confusion matrix at tuned threshold:


,pred_0,pred_1
actual_0,9818,5694
actual_1,10249,6374


This result says:

Threshold tuning helped validation,
but did not help test much.

Important comparison:

Best RF validation balanced accuracy: 53.25%
Best RF test balanced accuracy:       50.82%

So the tuned threshold 0.53 is not very reliable on test.

That means for this outperform target, we should not depend on hard prediction:

0 = not outperform
1 = outperform

Instead, we should check whether the model is good for ranking stocks. That is more useful

### Best Random Forest decile analysis

In [20]:
BEST_RF_DECILE_REPORT_PATH = REPORT_DIR / "best_rf_decile_analysis_v1.csv"

def probability_decile_analysis(eval_df: pd.DataFrame, proba: np.ndarray, split_name: str) -> pd.DataFrame:
    """
    Check whether higher model probability groups have better actual outcomes.

    Decile 10 = highest predicted probability group.
    Decile 1 = lowest predicted probability group.
    """
    temp = eval_df.copy()
    temp["predicted_probability"] = proba

    # Rank first to avoid qcut issues when many probabilities are similar.
    temp["probability_rank"] = temp["predicted_probability"].rank(method="first")

    temp["probability_decile"] = pd.qcut(
        temp["probability_rank"],
        q=10,
        labels=False,
    ) + 1

    summary = (
        temp
        .groupby("probability_decile")
        .agg(
            rows=("target_outperform_nifty50_20d", "size"),
            avg_predicted_probability=("predicted_probability", "mean"),
            actual_outperform_rate=("target_outperform_nifty50_20d", "mean"),
            avg_future_return_20d=("future_return_20d", "mean"),
            avg_nifty50_future_return_20d=("future_nifty50_return_20d", "mean"),
            avg_excess_return_vs_nifty50=("future_excess_return_20d_vs_nifty50", "mean"),
            downside_5pct_rate=("target_downside_5pct_20d", "mean"),
            big_downside_10pct_rate=("target_big_downside_10pct_20d", "mean"),
        )
        .reset_index()
        .sort_values("probability_decile", ascending=False)
    )

    summary["split"] = split_name

    return summary


best_rf_valid_proba = best_rf_model.predict_proba(X_valid)[:, 1]
best_rf_test_proba = best_rf_model.predict_proba(X_test)[:, 1]


valid_eval_df = model_data.loc[
    valid_mask,
    [
        "date",
        "symbol",
        "yf_ticker",
        "future_return_20d",
        "future_nifty50_return_20d",
        "future_excess_return_20d_vs_nifty50",
        "target_outperform_nifty50_20d",
        "target_downside_5pct_20d",
        "target_big_downside_10pct_20d",
    ]
].copy()

test_eval_df = model_data.loc[
    test_mask,
    [
        "date",
        "symbol",
        "yf_ticker",
        "future_return_20d",
        "future_nifty50_return_20d",
        "future_excess_return_20d_vs_nifty50",
        "target_outperform_nifty50_20d",
        "target_downside_5pct_20d",
        "target_big_downside_10pct_20d",
    ]
].copy()


best_rf_valid_decile_summary = probability_decile_analysis(
    eval_df=valid_eval_df,
    proba=best_rf_valid_proba,
    split_name="valid",
)

best_rf_test_decile_summary = probability_decile_analysis(
    eval_df=test_eval_df,
    proba=best_rf_test_proba,
    split_name="test",
)

best_rf_decile_summary = pd.concat(
    [best_rf_valid_decile_summary, best_rf_test_decile_summary],
    ignore_index=True,
)

print("Best Random Forest validation decile summary:")
display(best_rf_valid_decile_summary)

print("\nBest Random Forest test decile summary:")
display(best_rf_test_decile_summary)

print("\nTop decile vs bottom decile:")

best_rf_top_bottom_summary = best_rf_decile_summary[
    best_rf_decile_summary["probability_decile"].isin([1, 10])
].copy()

display(best_rf_top_bottom_summary)

best_rf_decile_summary.to_csv(BEST_RF_DECILE_REPORT_PATH, index=False)

print("\nSaved:")
print(BEST_RF_DECILE_REPORT_PATH)

Best Random Forest validation decile summary:


,probability_decile,rows,avg_predicted_probability,actual_outperform_rate,avg_future_return_20d,avg_nifty50_future_return_20d,avg_excess_return_vs_nifty50,downside_5pct_rate,big_downside_10pct_rate,split
9,10,2178,0.572424,0.578972,0.036113,0.012405,0.023708,0.337466,0.113407,valid
8,9,2178,0.545921,0.564738,0.027228,0.009776,0.017452,0.362259,0.110193,valid
7,8,2178,0.534167,0.533058,0.016636,0.005812,0.010824,0.375574,0.124426,valid
6,7,2178,0.524826,0.490817,0.009094,0.005898,0.003196,0.400367,0.141873,valid
5,6,2178,0.516740,0.488981,0.008636,0.005725,0.002911,0.379706,0.109275,valid
4,5,2178,0.509398,0.498163,0.010244,0.004478,0.005766,0.395776,0.117998,valid
3,4,2178,0.502332,0.480716,0.005604,0.002389,0.003215,0.419651,0.118916,valid
2,3,2178,0.494679,0.471534,0.005870,0.001967,0.003903,0.421028,0.141414,valid
1,2,2178,0.484683,0.46832,0.009863,0.007701,0.002162,0.399908,0.140496,valid
0,1,2178,0.464441,0.470156,0.013865,0.012220,0.001645,0.412305,0.136364,valid



Best Random Forest test decile summary:


,probability_decile,rows,avg_predicted_probability,actual_outperform_rate,avg_future_return_20d,avg_nifty50_future_return_20d,avg_excess_return_vs_nifty50,downside_5pct_rate,big_downside_10pct_rate,split
9,10,3214,0.600305,0.545426,0.024265,0.008800,0.015369,0.256689,0.068139,test
8,9,3213,0.568698,0.505135,0.006234,-0.001869,0.007810,0.342982,0.111734,test
7,8,3214,0.548234,0.540759,0.010440,-0.000080,0.010437,0.331985,0.105476,test
6,7,3213,0.533419,0.526299,0.005786,-0.001413,0.007089,0.348895,0.104575,test
5,6,3213,0.521773,0.519141,0.007682,0.000603,0.006882,0.35263,0.101774,test
4,5,3214,0.510764,0.518979,0.006079,0.000211,0.005646,0.381456,0.120411,test
3,4,3213,0.500090,0.540616,0.008553,-0.001530,0.009745,0.357921,0.113601,test
2,3,3214,0.489996,0.52458,0.007858,-0.000025,0.007784,0.368699,0.127567,test
1,2,3213,0.477856,0.475568,0.007107,0.005290,0.001420,0.389045,0.136632,test
0,1,3214,0.449700,0.476353,0.018032,0.014185,0.002779,0.394835,0.129745,test



Top decile vs bottom decile:


,probability_decile,rows,avg_predicted_probability,actual_outperform_rate,avg_future_return_20d,avg_nifty50_future_return_20d,avg_excess_return_vs_nifty50,downside_5pct_rate,big_downside_10pct_rate,split
0,10,2178,0.572424,0.578972,0.036113,0.012405,0.023708,0.337466,0.113407,valid
9,1,2178,0.464441,0.470156,0.013865,0.012220,0.001645,0.412305,0.136364,valid
10,10,3214,0.600305,0.545426,0.024265,0.008800,0.015369,0.256689,0.068139,test
19,1,3214,0.449700,0.476353,0.018032,0.014185,0.002779,0.394835,0.129745,test



Saved:
E:\Projects\marketguard-india\reports\baseline_modeling\best_rf_decile_analysis_v1.csv


## Best Random Forest Decile Analysis Summary

The Random Forest model is trained to predict:

`target_outperform_nifty50_20d`

This target means:

- `1`: The stock outperformed Nifty 50 over the next 20 trading days.
- `0`: The stock did not outperform Nifty 50 over the next 20 trading days.

Instead of only checking the model as a hard `0/1` classifier, we also evaluate whether the model is useful for ranking stocks.

For each stock-date row, the model outputs a probability of outperforming Nifty 50. We sort all rows by this predicted probability and split them into 10 equal groups called deciles.

- Decile 10 = top 10% highest predicted probabilities.
- Decile 1 = bottom 10% lowest predicted probabilities.

The goal is to check whether the stocks ranked highest by the model actually performed better than the stocks ranked lowest.

### Columns in the decile table

| Column | Meaning |
|---|---|
| `probability_decile` | Ranking group based on predicted probability. Decile 10 is highest confidence; Decile 1 is lowest confidence. |
| `rows` | Number of stock-date rows in that decile. |
| `avg_predicted_probability` | Average model probability for that decile. |
| `actual_outperform_rate` | Percentage of rows that actually outperformed Nifty 50 over the next 20 trading days. |
| `avg_future_return_20d` | Average stock return over the next 20 trading days. |
| `avg_nifty50_future_return_20d` | Average Nifty 50 return over the same next 20 trading days. |
| `avg_excess_return_vs_nifty50` | Average stock return minus average Nifty 50 return. This shows relative performance. |
| `downside_5pct_rate` | Percentage of rows where the stock fell at least 5% at any point in the next 20 trading days. |
| `big_downside_10pct_rate` | Percentage of rows where the stock fell at least 10% at any point in the next 20 trading days. |
| `split` | Whether the result belongs to validation data or test data. |

### How the comparison is done

The main comparison is between:

- Decile 10: stocks the model liked the most.
- Decile 1: stocks the model liked the least.

If the model is useful, Decile 10 should have:

- Higher actual outperform rate.
- Higher average excess return vs Nifty 50.
- Lower downside risk.

### Validation result

In validation data:

| Metric | Decile 10 | Decile 1 |
|---|---:|---:|
| Actual outperform rate | 57.90% | 47.02% |
| Avg excess return vs Nifty 50 | +2.37% | +0.16% |
| 5% downside rate | 33.75% | 41.23% |
| 10% downside rate | 11.34% | 13.64% |

This shows that the model's top-ranked group performed better than the bottom-ranked group.

### Test result

In test data:

| Metric | Decile 10 | Decile 1 |
|---|---:|---:|
| Actual outperform rate | 54.54% | 47.64% |
| Avg excess return vs Nifty 50 | +1.54% | +0.28% |
| 5% downside rate | 25.67% | 39.48% |
| 10% downside rate | 6.81% | 12.97% |

The test result confirms that the model ranking has useful signal.

### Interpretation

The Random Forest is weak as a strict binary classifier because the accuracy and threshold-based results are not strong enough.

However, the decile analysis shows that the model is useful as a ranking model.

Higher predicted probability generally corresponds to:

- Better chance of outperforming Nifty 50.
- Higher excess return vs Nifty 50.
- Lower downside risk.

Therefore, this model should not be used as a simple `buy / don't buy` or `outperform / not outperform` classifier.

Instead, it is better suited for ranking stocks from stronger to weaker based on their predicted probability of outperforming Nifty 50.

### Final conclusion

The tuned Random Forest model provides a useful baseline ranking signal.

For the MarketGuard India dashboard, this model can be used to rank stocks by relative strength probability, but it should be combined with separate downside-risk models before making any final risk score or decision-support output.

### Stronger model: HistGradientBoosting

lets try stronger boosting model and see if it improves anything 

In [21]:
from sklearn.ensemble import HistGradientBoostingClassifier

hist_gb_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            HistGradientBoostingClassifier(
                max_iter=300,
                learning_rate=0.05,
                max_leaf_nodes=31,
                min_samples_leaf=100,
                l2_regularization=0.10,
                early_stopping=True,
                validation_fraction=0.10,
                random_state=42,
            ),
        ),
    ]
)


In [22]:

hist_gb_model.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](79,)","['return_1d','return_5d','return_20d',...,'vix_above_20_flag', 'vix_above_25_flag','vix_20d_change']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,79
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_valu

In [23]:

baseline_models["hist_gradient_boosting"] = hist_gb_model

hist_gb_results = []

for split_name, X, y in [
    ("train", X_train, y_train),
    ("valid", X_valid, y_valid),
    ("test", X_test, y_test),
]:
    hist_gb_results.append(
        evaluate_classifier(
            model=hist_gb_model,
            X=X,
            y=y,
            split_name=split_name,
            model_name="hist_gradient_boosting",
        )
    )

hist_gb_results_df = pd.DataFrame(hist_gb_results)

display(hist_gb_results_df)

print("\nValidation confusion matrix:")

hist_gb_valid_pred = hist_gb_model.predict(X_valid)

display(
    pd.DataFrame(
        confusion_matrix(y_valid, hist_gb_valid_pred),
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
)

print("\nTest confusion matrix:")

hist_gb_test_pred = hist_gb_model.predict(X_test)

display(
    pd.DataFrame(
        confusion_matrix(y_test, hist_gb_test_pred),
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
)

,model,split,rows,positive_rate_actual,positive_rate_predicted,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,hist_gradient_boosting,train,253538,0.512925,0.545449,0.697884,0.696840,0.693243,0.737201,0.714547,0.770659,0.776076
1,hist_gradient_boosting,valid,21780,0.504545,0.590542,0.525941,0.525120,0.525812,0.615434,0.567104,0.530856,0.525802
2,hist_gradient_boosting,test,32135,0.517286,0.639832,0.527991,0.523185,0.535383,0.662215,0.592083,0.525463,0.531033



Validation confusion matrix:


,pred_0,pred_1
actual_0,4692,6099
actual_1,4226,6763



Test confusion matrix:


,pred_0,pred_1
actual_0,5959,9553
actual_1,5615,11008


Here the model is overfitting 

The main evidence is:

Train ROC AUC:  0.7707
Valid ROC AUC:  0.5309
Test ROC AUC:   0.5255

That means the model learned the training period very well, but the signal does not generalize strongly to future years.

But it is not completely useless:

Best RF test ROC AUC:      0.5179
HistGB test ROC AUC:       0.5255

So HistGradientBoosting did slightly better than RF on test, but worse than RF on validation:

Best RF valid ROC AUC:     0.5404
HistGB valid ROC AUC:      0.5309

The likely issue is:

Bigger model learns historical patterns,
but those patterns are unstable across market regimes.

Next, try a more regularized HistGB. This makes the model less aggressive.

In [25]:
# Prepare best Random Forest metrics for later model comparison

best_rf_final_results = []

for split_name, X, y in [
    ("train", X_train, y_train),
    ("valid", X_valid, y_valid),
    ("test", X_test, y_test),
]:
    best_rf_final_results.append(
        evaluate_classifier(
            model=best_rf_model,
            X=X,
            y=y,
            split_name=split_name,
            model_name="best_random_forest",
        )
    )

best_rf_final_results_df = pd.DataFrame(best_rf_final_results)

display(best_rf_final_results_df)

,model,split,rows,positive_rate_actual,positive_rate_predicted,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,best_random_forest,train,253538,0.512925,0.570226,0.645465,0.643746,0.638884,0.710256,0.672682,0.700786,0.699147
1,best_random_forest,valid,21780,0.504545,0.681635,0.524931,0.523282,0.521622,0.704705,0.599497,0.540441,0.539940
2,best_random_forest,test,32135,0.517286,0.650350,0.519216,0.514035,0.528064,0.663899,0.588242,0.517855,0.530662


### More regularized HistGradientBoosting

In [26]:
hist_gb_regularized_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            HistGradientBoostingClassifier(
                max_iter=150,
                learning_rate=0.03,
                max_leaf_nodes=15,
                min_samples_leaf=300,
                l2_regularization=1.0,
                early_stopping=True,
                validation_fraction=0.10,
                random_state=42,
            ),
        ),
    ]
)

hist_gb_regularized_model.fit(X_train, y_train)

baseline_models["hist_gradient_boosting_regularized"] = hist_gb_regularized_model

hist_gb_regularized_results = []

for split_name, X, y in [
    ("train", X_train, y_train),
    ("valid", X_valid, y_valid),
    ("test", X_test, y_test),
]:
    hist_gb_regularized_results.append(
        evaluate_classifier(
            model=hist_gb_regularized_model,
            X=X,
            y=y,
            split_name=split_name,
            model_name="hist_gradient_boosting_regularized",
        )
    )

hist_gb_regularized_results_df = pd.DataFrame(hist_gb_regularized_results)

display(hist_gb_regularized_results_df)

print("\nComparison:")

model_comparison_df = pd.concat(
    [
        best_rf_final_results_df,
        hist_gb_results_df,
        hist_gb_regularized_results_df,
    ],
    ignore_index=True,
)

display(
    model_comparison_df[
        ["model", "split", "accuracy", "balanced_accuracy", "roc_auc", "average_precision"]
    ].sort_values(["split", "roc_auc"], ascending=[True, False])
)

,model,split,rows,positive_rate_actual,positive_rate_predicted,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,hist_gradient_boosting_regularized,train,253538,0.512925,0.617513,0.603330,0.600359,0.594131,0.715278,0.649100,0.650046,0.654596
1,hist_gradient_boosting_regularized,valid,21780,0.504545,0.709596,0.524059,0.522155,0.520155,0.731550,0.608002,0.541513,0.532401
2,hist_gradient_boosting_regularized,test,32135,0.517286,0.713708,0.526684,0.519319,0.530804,0.732359,0.615501,0.522465,0.526740



Comparison:


,model,split,accuracy,balanced_accuracy,roc_auc,average_precision
5,hist_gradient_boosting,test,0.527991,0.523185,0.525463,0.531033
8,hist_gradient_boosting_regularized,test,0.526684,0.519319,0.522465,0.526740
2,best_random_forest,test,0.519216,0.514035,0.517855,0.530662
3,hist_gradient_boosting,train,0.697884,0.696840,0.770659,0.776076
0,best_random_forest,train,0.645465,0.643746,0.700786,0.699147
6,hist_gradient_boosting_regularized,train,0.603330,0.600359,0.650046,0.654596
7,hist_gradient_boosting_regularized,valid,0.524059,0.522155,0.541513,0.532401
1,best_random_forest,valid,0.524931,0.523282,0.540441,0.539940
4,hist_gradient_boosting,valid,0.525941,0.525120,0.530856,0.525802


The regularized model reduced overfitting, but it did not clearly beat everything.

What changed
Original HistGB train ROC AUC:      0.7707
Regularized HistGB train ROC AUC:   0.6500

So regularization worked. The model became less memorizing.

Since this target is for ranking, we need to compare their decile quality.

### Compare decile ranking across RF and HistGB models

In [27]:
MODEL_DECILE_COMPARISON_PATH = REPORT_DIR / "model_decile_comparison_outperform_nifty50_20d_v1.csv"

def get_top_bottom_decile_summary(model, model_name, X, eval_df, split_name):
    proba = model.predict_proba(X)[:, 1]

    decile_summary = probability_decile_analysis(
        eval_df=eval_df,
        proba=proba,
        split_name=split_name,
    )

    top_bottom = decile_summary[
        decile_summary["probability_decile"].isin([1, 10])
    ].copy()

    top_bottom["model"] = model_name

    return top_bottom


decile_comparison_parts = []

for model_name, model in [
    ("best_random_forest", best_rf_model),
    ("hist_gradient_boosting", hist_gb_model),
    ("hist_gradient_boosting_regularized", hist_gb_regularized_model),
]:
    decile_comparison_parts.append(
        get_top_bottom_decile_summary(
            model=model,
            model_name=model_name,
            X=X_valid,
            eval_df=valid_eval_df,
            split_name="valid",
        )
    )

    decile_comparison_parts.append(
        get_top_bottom_decile_summary(
            model=model,
            model_name=model_name,
            X=X_test,
            eval_df=test_eval_df,
            split_name="test",
        )
    )

model_decile_comparison_df = pd.concat(
    decile_comparison_parts,
    ignore_index=True,
)

display(
    model_decile_comparison_df[
        [
            "model",
            "split",
            "probability_decile",
            "rows",
            "avg_predicted_probability",
            "actual_outperform_rate",
            "avg_excess_return_vs_nifty50",
            "downside_5pct_rate",
            "big_downside_10pct_rate",
        ]
    ].sort_values(["split", "model", "probability_decile"], ascending=[True, True, False])
)

model_decile_comparison_df.to_csv(MODEL_DECILE_COMPARISON_PATH, index=False)

print("Saved:")
print(MODEL_DECILE_COMPARISON_PATH)

,model,split,probability_decile,rows,avg_predicted_probability,actual_outperform_rate,avg_excess_return_vs_nifty50,downside_5pct_rate,big_downside_10pct_rate
2,best_random_forest,test,10,3214,0.600305,0.545426,0.015369,0.256689,0.068139
3,best_random_forest,test,1,3214,0.449700,0.476353,0.002779,0.394835,0.129745
6,hist_gradient_boosting,test,10,3214,0.652493,0.519291,0.010364,0.336341,0.086185
7,hist_gradient_boosting,test,1,3214,0.393581,0.439017,-0.003007,0.379589,0.130989
10,hist_gradient_boosting_regularized,test,10,3214,0.586671,0.523024,0.010704,0.291848,0.071251
11,hist_gradient_boosting_regularized,test,1,3214,0.452039,0.462974,0.000449,0.393902,0.1285
0,best_random_forest,valid,10,2178,0.572424,0.578972,0.023708,0.337466,0.113407
1,best_random_forest,valid,1,2178,0.464441,0.470156,0.001645,0.412305,0.136364
4,hist_gradient_boosting,valid,10,2178,0.634155,0.541781,0.016774,0.314968,0.105142
5,hist_gradient_boosting,valid,1,2178,0.392070,0.469238,0.004273,0.401286,0.117998


Saved:
E:\Projects\marketguard-india\reports\baseline_modeling\model_decile_comparison_outperform_nifty50_20d_v1.csv


## Model Comparison Summary: Random Forest vs HistGradientBoosting

After completing the tuned Random Forest baseline, we tested stronger Gradient Boosting models to see whether a bigger model could improve performance on the `target_outperform_nifty50_20d` target.

The target is:

`target_outperform_nifty50_20d`

This means:

- `1`: The stock outperformed Nifty 50 over the next 20 trading days.
- `0`: The stock did not outperform Nifty 50 over the next 20 trading days.

The main models compared were:

1. `best_random_forest`
2. `hist_gradient_boosting`
3. `hist_gradient_boosting_regularized`

### Classification Metric Comparison

| Model | Split | ROC AUC | Average Precision |
|---|---|---:|---:|
| `best_random_forest` | Validation | 0.5404 | 0.5399 |
| `best_random_forest` | Test | 0.5179 | 0.5307 |
| `hist_gradient_boosting` | Validation | 0.5309 | 0.5258 |
| `hist_gradient_boosting` | Test | 0.5255 | 0.5310 |
| `hist_gradient_boosting_regularized` | Validation | 0.5415 | 0.5324 |
| `hist_gradient_boosting_regularized` | Test | 0.5225 | 0.5267 |

The original HistGradientBoosting model overfit the training data:

| Model | Train ROC AUC | Validation ROC AUC | Test ROC AUC |
|---|---:|---:|---:|
| `hist_gradient_boosting` | 0.7707 | 0.5309 | 0.5255 |
| `hist_gradient_boosting_regularized` | 0.6500 | 0.5415 | 0.5225 |

The regularized version reduced overfitting, but it did not clearly dominate the other models on test data.

### Decile Ranking Comparison

Since the classification metrics were weak, we compared the models using decile analysis.

For each model:

- All rows were sorted by predicted probability.
- The top 10% highest-probability rows were placed in Decile 10.
- The bottom 10% lowest-probability rows were placed in Decile 1.

A useful ranking model should show:

- Decile 10 outperform rate higher than Decile 1.
- Decile 10 excess return higher than Decile 1.
- Decile 10 downside risk lower than Decile 1.

### Test Split: Top vs Bottom Decile

#### Best Random Forest

| Metric | Decile 10 | Decile 1 | Difference |
|---|---:|---:|---:|
| Actual outperform rate | 54.54% | 47.64% | +6.91 percentage points |
| Avg excess return vs Nifty 50 | +1.54% | +0.28% | +1.26% |
| 5% downside rate | 25.67% | 39.48% | 13.81 percentage points lower risk |
| 10% downside rate | 6.81% | 12.97% | 6.16 percentage points lower risk |

#### HistGradientBoosting

| Metric | Decile 10 | Decile 1 | Difference |
|---|---:|---:|---:|
| Actual outperform rate | 51.93% | 43.90% | +8.03 percentage points |
| Avg excess return vs Nifty 50 | +1.04% | -0.30% | +1.34% |
| 5% downside rate | 33.63% | 37.96% | 4.32 percentage points lower risk |
| 10% downside rate | 8.62% | 13.10% | 4.48 percentage points lower risk |

#### Regularized HistGradientBoosting

| Metric | Decile 10 | Decile 1 | Difference |
|---|---:|---:|---:|
| Actual outperform rate | 52.30% | 46.30% | +6.01 percentage points |
| Avg excess return vs Nifty 50 | +1.07% | +0.04% | +1.03% |
| 5% downside rate | 29.18% | 39.39% | 10.21 percentage points lower risk |
| 10% downside rate | 7.13% | 12.85% | 5.72 percentage points lower risk |

### Interpretation

The stronger HistGradientBoosting models improved some metrics, but they did not clearly beat the tuned Random Forest.

The HistGradientBoosting model had the best test ROC AUC, but it showed stronger overfitting. The regularized HistGradientBoosting model reduced overfitting and had the best validation ROC AUC, but its test performance was not clearly superior.

The tuned Random Forest remained the most balanced model because it gave:

- Good validation performance.
- Competitive test performance.
- Strong top-decile vs bottom-decile separation.
- Better downside-risk separation than the boosting models.

### Final Conclusion

For the `target_outperform_nifty50_20d` target, bigger models did not provide a strong enough improvement to replace Random Forest.

The best current baseline for this target is:

`best_random_forest`

This model should not be used as a hard `outperform / not outperform` classifier. Instead, it should be used as a ranking model.

For the MarketGuard India dashboard, the Random Forest outperform model can rank stocks by relative strength probability, but it should be combined with a separate downside-risk model before creating a final risk score or decision-support signal.

The next important modeling target is:

`target_big_downside_10pct_20d`

This target is more directly aligned with MarketGuard's purpose because it predicts whether a stock may fall at least 10% at any point over the next 20 trading days.

### Save Selected Outperform Model and Final Evaluation Summary

In [28]:
import joblib

BEST_RF_MODEL_PATH = MODEL_DIR / "best_random_forest_outperform_nifty50_20d_v1.joblib"
BEST_RF_FINAL_REPORT_PATH = REPORT_DIR / "best_random_forest_final_summary_v1.csv"

# Recompute final train/valid/test metrics for saved report
best_rf_final_results = []

for split_name, X, y in [
    ("train", X_train, y_train),
    ("valid", X_valid, y_valid),
    ("test", X_test, y_test),
]:
    best_rf_final_results.append(
        evaluate_classifier(
            model=best_rf_model,
            X=X,
            y=y,
            split_name=split_name,
            model_name="best_random_forest",
        )
    )

best_rf_final_results_df = pd.DataFrame(best_rf_final_results)

# Add chosen threshold metrics
best_rf_threshold_final_df["report_type"] = "threshold_tuned"
best_rf_final_results_df["report_type"] = "default_threshold"

combined_best_rf_summary = pd.concat(
    [
        best_rf_final_results_df,
        best_rf_threshold_final_df,
    ],
    ignore_index=True,
    sort=False,
)

display(combined_best_rf_summary)

# Save model
joblib.dump(best_rf_model, BEST_RF_MODEL_PATH)

# Save summary
combined_best_rf_summary.to_csv(BEST_RF_FINAL_REPORT_PATH, index=False)

print("Saved model:")
print(BEST_RF_MODEL_PATH)

print("\nSaved final RF summary:")
print(BEST_RF_FINAL_REPORT_PATH)

print("\nAlready saved decile report:")
print(BEST_RF_DECILE_REPORT_PATH)

,model,split,rows,positive_rate_actual,positive_rate_predicted,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,report_type,threshold
0,best_random_forest,train,253538.0,0.512925,0.570226,0.645465,0.643746,0.638884,0.710256,0.672682,0.700786,0.699147,default_threshold,NaN
1,best_random_forest,valid,21780.0,0.504545,0.681635,0.524931,0.523282,0.521622,0.704705,0.599497,0.540441,0.539940,default_threshold,NaN
2,best_random_forest,test,32135.0,0.517286,0.650350,0.519216,0.514035,0.528064,0.663899,0.588242,0.517855,0.530662,default_threshold,NaN
3,best_random_forest,valid,NaN,NaN,0.292011,0.530624,0.532518,0.560220,0.324233,0.410744,NaN,NaN,threshold_tuned,0.53
4,best_random_forest,test,NaN,NaN,0.375541,0.503874,0.508187,0.528174,0.383445,0.444321,NaN,NaN,threshold_tuned,0.53


Saved model:
E:\Projects\marketguard-india\models\best_random_forest_outperform_nifty50_20d_v1.joblib

Saved final RF summary:
E:\Projects\marketguard-india\reports\baseline_modeling\best_random_forest_final_summary_v1.csv

Already saved decile report:
E:\Projects\marketguard-india\reports\baseline_modeling\best_rf_decile_analysis_v1.csv


### Save Outperform Feature List and Model Metadata

In [29]:
import json
from datetime import datetime

OUTPERFORM_FEATURE_LIST_PATH = (
    REPORT_DIR / "outperform_nifty50_20d_feature_list_v1.csv"
)

OUTPERFORM_MODEL_METADATA_PATH = (
    REPORT_DIR / "outperform_nifty50_20d_model_metadata_v1.json"
)

# Save exact feature order
outperform_feature_list_df = pd.DataFrame(
    {
        "feature_order": range(1, len(feature_cols) + 1),
        "feature": feature_cols,
    }
)

outperform_feature_list_df.to_csv(
    OUTPERFORM_FEATURE_LIST_PATH,
    index=False,
)

# Save model metadata
outperform_model_metadata = {
    "model_name": "best_random_forest",
    "model_type": "RandomForestClassifier",
    "model_version": "v1",
    "model_path": str(BEST_RF_MODEL_PATH),
    "target": "target_outperform_nifty50_20d",
    "target_meaning": (
        "1 means the stock outperformed Nifty 50 over the "
        "next 20 trading days."
    ),
    "intended_use": (
        "Rank stocks by estimated probability of outperforming "
        "Nifty 50. Not intended as a hard buy or sell classifier."
    ),
    "feature_count": len(feature_cols),
    "train_rows": int(len(X_train)),
    "valid_rows": int(len(X_valid)),
    "test_rows": int(len(X_test)),
    "train_positive_rate": float(y_train.mean()),
    "valid_positive_rate": float(y_valid.mean()),
    "test_positive_rate": float(y_test.mean()),
    "default_threshold": 0.50,
    "selected_threshold": 0.53,
    "model_parameters": {
        "n_estimators": 300,
        "min_samples_split": 200,
        "min_samples_leaf": 400,
        "max_features": 0.7,
        "max_depth": 10,
        "class_weight": None,
        "bootstrap": True,
        "random_state": 42,
    },
    "performance": {
        "validation_roc_auc": 0.540441,
        "validation_average_precision": 0.539940,
        "test_roc_auc": 0.517855,
        "test_average_precision": 0.530662,
    },
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

with open(
    OUTPERFORM_MODEL_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(outperform_model_metadata, file, indent=4)

print("Saved outperform model:")
print(BEST_RF_MODEL_PATH)

print("\nSaved exact feature list:")
print(OUTPERFORM_FEATURE_LIST_PATH)

print("\nSaved model metadata:")
print(OUTPERFORM_MODEL_METADATA_PATH)

print("\nFeature count:", len(feature_cols))

Saved outperform model:
E:\Projects\marketguard-india\models\best_random_forest_outperform_nifty50_20d_v1.joblib

Saved exact feature list:
E:\Projects\marketguard-india\reports\baseline_modeling\outperform_nifty50_20d_feature_list_v1.csv

Saved model metadata:
E:\Projects\marketguard-india\reports\baseline_modeling\outperform_nifty50_20d_model_metadata_v1.json

Feature count: 79


Next we move to the main MarketGuard risk target:

target_big_downside_10pct_20d

Meaning:

1 = stock fell at least 10% at any point in the next 20 trading days
0 = stock did not fall 10% in the next 20 trading days

We will use the same features and same train/valid/test split, but change the target.

### Prepare 20-day big downside target

In [30]:
DOWNSIDE_TARGET_COL = "target_big_downside_10pct_20d"

# Use the same model_data, same time split, and same X_train/X_valid/X_test.
# Only the target changes.
y_train_downside = model_data.loc[train_mask, DOWNSIDE_TARGET_COL].astype(int)
y_valid_downside = model_data.loc[valid_mask, DOWNSIDE_TARGET_COL].astype(int)
y_test_downside = model_data.loc[test_mask, DOWNSIDE_TARGET_COL].astype(int)

print("Downside target:", DOWNSIDE_TARGET_COL)

print("\nTarget meaning:")
print("1 = stock fell at least 10% at any point in the next 20 trading days")
print("0 = stock did not fall 10% in the next 20 trading days")

print("\nShapes:")
print("X_train:", X_train.shape, "y_train_downside:", y_train_downside.shape)
print("X_valid:", X_valid.shape, "y_valid_downside:", y_valid_downside.shape)
print("X_test :", X_test.shape,  "y_test_downside :", y_test_downside.shape)

print("\nPositive rate by split:")
print("Train:", round(y_train_downside.mean() * 100, 2), "%")
print("Valid:", round(y_valid_downside.mean() * 100, 2), "%")
print("Test :", round(y_test_downside.mean() * 100, 2), "%")

print("\nClass counts:")

for split_name, y in [
    ("train", y_train_downside),
    ("valid", y_valid_downside),
    ("test", y_test_downside),
]:
    print(f"\n{split_name}:")
    display(
        y.value_counts()
        .sort_index()
        .rename(index={0: "0_no_big_downside", 1: "1_big_downside"})
        .to_frame("count")
    )

Downside target: target_big_downside_10pct_20d

Target meaning:
1 = stock fell at least 10% at any point in the next 20 trading days
0 = stock did not fall 10% in the next 20 trading days

Shapes:
X_train: (253538, 79) y_train_downside: (253538,)
X_valid: (21780, 79) y_valid_downside: (21780,)
X_test : (32135, 79) y_test_downside : (32135,)

Positive rate by split:
Train: 14.76 %
Valid: 12.54 %
Test : 11.2 %

Class counts:

train:


,count
target_big_downside_10pct_20d,
0_no_big_downside,216116
1_big_downside,37422



valid:


,count
target_big_downside_10pct_20d,
0_no_big_downside,19048
1_big_downside,2732



test:


,count
target_big_downside_10pct_20d,
0_no_big_downside,28537
1_big_downside,3598


This target is clearly imbalanced:

Train big downside rate: 14.76%

Valid big downside rate: 12.54%

Test big downside rate:  11.20%

So an all-0 dummy model can get around 88.8% test accuracy, but it detects zero actual downside events. For this target, accuracy is not the main metric.

We care more about:

ROC AUC
Average precision
Recall for class 1
Precision for class 1

### Downside risk baseline models

In [31]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

SAFE_N_JOBS = 8

DOWNSIDE_BASELINE_REPORT_PATH = REPORT_DIR / "downside_10pct_baseline_results_v1.csv"

downside_models = {}

# Dummy baseline: predicts majority class, usually 0_no_big_downside
downside_dummy_model = DummyClassifier(strategy="most_frequent")
downside_dummy_model.fit(X_train, y_train_downside)

downside_models["dummy_most_frequent_downside"] = downside_dummy_model


In [32]:


# Logistic Regression baseline with class balancing
downside_logistic_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                solver="lbfgs",
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

downside_logistic_model.fit(X_train, y_train_downside)

downside_models["logistic_regression_downside_balanced"] = downside_logistic_model



In [33]:

# Random Forest downside model
downside_rf_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=200,
                min_samples_split=200,
                max_features="sqrt",
                class_weight="balanced_subsample",
                bootstrap=True,
                random_state=42,
                n_jobs=SAFE_N_JOBS,
            ),
        ),
    ]
)

downside_rf_model.fit(X_train, y_train_downside)

downside_models["random_forest_downside_balanced"] = downside_rf_model


In [34]:


downside_results = []

for model_name, model in downside_models.items():
    for split_name, X, y in [
        ("train", X_train, y_train_downside),
        ("valid", X_valid, y_valid_downside),
        ("test", X_test, y_test_downside),
    ]:
        downside_results.append(
            evaluate_classifier(
                model=model,
                X=X,
                y=y,
                split_name=split_name,
                model_name=model_name,
            )
        )

downside_results_df = pd.DataFrame(downside_results)

display(
    downside_results_df
    .sort_values(["model", "split"])
    .reset_index(drop=True)
)

downside_results_df.to_csv(DOWNSIDE_BASELINE_REPORT_PATH, index=False)

print("Saved:")
print(DOWNSIDE_BASELINE_REPORT_PATH)

print("\nValidation confusion matrix: Random Forest downside")

y_valid_downside_pred = downside_rf_model.predict(X_valid)

display(
    pd.DataFrame(
        confusion_matrix(y_valid_downside, y_valid_downside_pred),
        index=["actual_0_no_big_downside", "actual_1_big_downside"],
        columns=["pred_0_no_big_downside", "pred_1_big_downside"],
    )
)

print("\nTest confusion matrix: Random Forest downside")

y_test_downside_pred = downside_rf_model.predict(X_test)

display(
    pd.DataFrame(
        confusion_matrix(y_test_downside, y_test_downside_pred),
        index=["actual_0_no_big_downside", "actual_1_big_downside"],
        columns=["pred_0_no_big_downside", "pred_1_big_downside"],
    )
)

,model,split,rows,positive_rate_actual,positive_rate_predicted,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,dummy_most_frequent_downside,test,32135,0.111965,0.000000,0.888035,0.500000,0.000000,0.000000,0.000000,0.500000,0.111965
1,dummy_most_frequent_downside,train,253538,0.147599,0.000000,0.852401,0.500000,0.000000,0.000000,0.000000,0.500000,0.147599
2,dummy_most_frequent_downside,valid,21780,0.125436,0.000000,0.874564,0.500000,0.000000,0.000000,0.000000,0.500000,0.125436
3,logistic_regression_downside_balanced,test,32135,0.111965,0.327493,0.645745,0.529839,0.130084,0.380489,0.193882,0.552114,0.126520
4,logistic_regression_downside_balanced,train,253538,0.147599,0.364080,0.669762,0.646972,0.249177,0.614638,0.354598,0.704959,0.283024
5,logistic_regression_downside_balanced,valid,21780,0.125436,0.537879,0.484068,0.528359,0.137004,0.587482,0.222191,0.552220,0.150142
6,random_forest_downside_balanced,test,32135,0.111965,0.303781,0.704310,0.630823,0.197603,0.536131,0.288772,0.689434,0.210736
7,random_forest_downside_balanced,train,253538,0.147599,0.374358,0.703500,0.728408,0.301125,0.763749,0.431946,0.804967,0.448999
8,random_forest_downside_balanced,valid,21780,0.125436,0.278421,0.716713,0.615589,0.216524,0.480600,0.298545,0.655404,0.222705


Saved:
E:\Projects\marketguard-india\reports\baseline_modeling\downside_10pct_baseline_results_v1.csv

Validation confusion matrix: Random Forest downside


,pred_0_no_big_downside,pred_1_big_downside
actual_0_no_big_downside,14297,4751
actual_1_big_downside,1419,1313



Test confusion matrix: Random Forest downside


,pred_0_no_big_downside,pred_1_big_downside
actual_0_no_big_downside,20704,7833
actual_1_big_downside,1669,1929


This is a much better result than the outperform model.

For downside risk, Random Forest is clearly useful:

Dummy test ROC AUC:              0.5000
Logistic test ROC AUC:           0.5521
Random Forest test ROC AUC:      0.6894

And average precision improved a lot:

Test base downside rate:         11.20%
Random Forest avg precision:     21.07%

That means the model is finding risky cases better than random.

But the current threshold is aggressive:

Test predicted high-risk rate:   30.38%
Test recall:                     53.61%
Test precision:                  19.76%

Meaning:

It catches around 54% of actual 10% downside events,
but many alerts are false positives.

For our project, that is acceptable as a starting point because risk warning systems usually prefer catching danger, even with some false alarms.
Now we tune the threshold.

### Downside Random Forest threshold tuning

In [35]:
def evaluate_downside_threshold(y_true, y_proba, threshold: float) -> dict:
    """
    Evaluate downside-risk model at a custom probability threshold.

    Positive class:
    1 = stock falls at least 10% at any point in next 20 trading days
    """
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0

    return {
        "threshold": threshold,
        "alert_rate": y_pred.mean(),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "false_positive_rate": false_positive_rate,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn,
    }



In [36]:

downside_rf_valid_proba = downside_rf_model.predict_proba(X_valid)[:, 1]
downside_rf_test_proba = downside_rf_model.predict_proba(X_test)[:, 1]

downside_threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.01):
    downside_threshold_results.append(
        evaluate_downside_threshold(
            y_true=y_valid_downside,
            y_proba=downside_rf_valid_proba,
            threshold=threshold,
        )
    )

downside_threshold_results_df = pd.DataFrame(downside_threshold_results)

print("Top thresholds by validation F1:")
display(
    downside_threshold_results_df
    .sort_values("f1", ascending=False)
    .head(20)
)

print("\nTop thresholds by validation balanced accuracy:")
display(
    downside_threshold_results_df
    .sort_values("balanced_accuracy", ascending=False)
    .head(20)
)


best_downside_threshold_f1 = (
    downside_threshold_results_df
    .sort_values("f1", ascending=False)
    .iloc[0]["threshold"]
)

best_downside_threshold_balanced_accuracy = (
    downside_threshold_results_df
    .sort_values("balanced_accuracy", ascending=False)
    .iloc[0]["threshold"]
)

print("Best threshold by F1:", round(best_downside_threshold_f1, 2))
print("Best threshold by balanced accuracy:", round(best_downside_threshold_balanced_accuracy, 2))


# Evaluate selected thresholds on validation and test
selected_thresholds = [
    ("best_f1", best_downside_threshold_f1),
    ("best_balanced_accuracy", best_downside_threshold_balanced_accuracy),
    ("default_0_50", 0.50),
]

selected_threshold_results = []

for threshold_name, threshold in selected_thresholds:
    valid_metrics = evaluate_downside_threshold(
        y_true=y_valid_downside,
        y_proba=downside_rf_valid_proba,
        threshold=threshold,
    )

    test_metrics = evaluate_downside_threshold(
        y_true=y_test_downside,
        y_proba=downside_rf_test_proba,
        threshold=threshold,
    )

    selected_threshold_results.append(
        {
            "threshold_name": threshold_name,
            "split": "valid",
            **valid_metrics,
        }
    )

    selected_threshold_results.append(
        {
            "threshold_name": threshold_name,
            "split": "test",
            **test_metrics,
        }
    )

selected_downside_threshold_df = pd.DataFrame(selected_threshold_results)

display(selected_downside_threshold_df)

print("\nTest confusion matrix at best F1 threshold:")

best_f1_test_pred = (
    downside_rf_test_proba >= best_downside_threshold_f1
).astype(int)

display(
    pd.DataFrame(
        confusion_matrix(y_test_downside, best_f1_test_pred),
        index=["actual_0_no_big_downside", "actual_1_big_downside"],
        columns=["pred_0_no_big_downside", "pred_1_big_downside"],
    )
)

Top thresholds by validation F1:


,threshold,alert_rate,accuracy,balanced_accuracy,precision,recall,specificity,false_positive_rate,f1,true_positives,false_positives,false_negatives,true_negatives
41,0.51,0.246235,0.739440,0.612435,0.225620,0.442899,0.781972,0.218028,0.298950,1210,4153,1522,14895
40,0.50,0.278421,0.716713,0.615589,0.216524,0.480600,0.750577,0.249423,0.298545,1313,4751,1419,14297
39,0.49,0.312167,0.692883,0.618896,0.209001,0.520132,0.717661,0.282339,0.298185,1421,5378,1311,13670
42,0.52,0.215657,0.759826,0.606689,0.233979,0.402269,0.811109,0.188891,0.295868,1099,3598,1633,15450
38,0.48,0.346143,0.667631,0.619352,0.201088,0.554905,0.683799,0.316201,0.295200,1516,6023,1216,13025
37,0.47,0.382094,0.641139,0.620353,0.194545,0.592606,0.648100,0.351900,0.292926,1619,6703,1113,12345
43,0.53,0.186364,0.778926,0.600208,0.243410,0.361640,0.838776,0.161224,0.290973,988,3071,1744,15977
36,0.46,0.418595,0.612994,0.618528,0.187562,0.625915,0.611140,0.388860,0.288632,1710,7407,1022,11641
35,0.45,0.453627,0.586410,0.617752,0.182389,0.659590,0.575913,0.424087,0.285760,1802,8078,930,10970
44,0.54,0.159183,0.795546,0.591682,0.251803,0.319546,0.863818,0.136182,0.281658,873,2594,1859,16454



Top thresholds by validation balanced accuracy:


,threshold,alert_rate,accuracy,balanced_accuracy,precision,recall,specificity,false_positive_rate,f1,true_positives,false_positives,false_negatives,true_negatives
37,0.47,0.382094,0.641139,0.620353,0.194545,0.592606,0.648100,0.351900,0.292926,1619,6703,1113,12345
38,0.48,0.346143,0.667631,0.619352,0.201088,0.554905,0.683799,0.316201,0.295200,1516,6023,1216,13025
39,0.49,0.312167,0.692883,0.618896,0.209001,0.520132,0.717661,0.282339,0.298185,1421,5378,1311,13670
36,0.46,0.418595,0.612994,0.618528,0.187562,0.625915,0.611140,0.388860,0.288632,1710,7407,1022,11641
35,0.45,0.453627,0.586410,0.617752,0.182389,0.659590,0.575913,0.424087,0.285760,1802,8078,930,10970
40,0.50,0.278421,0.716713,0.615589,0.216524,0.480600,0.750577,0.249423,0.298545,1313,4751,1419,14297
41,0.51,0.246235,0.739440,0.612435,0.225620,0.442899,0.781972,0.218028,0.298950,1210,4153,1522,14895
34,0.44,0.491322,0.555785,0.612314,0.175591,0.687775,0.536854,0.463146,0.279759,1879,8822,853,10226
33,0.43,0.525712,0.528099,0.607930,0.170480,0.714495,0.501365,0.498635,0.275279,1952,9498,780,9550
42,0.52,0.215657,0.759826,0.606689,0.233979,0.402269,0.811109,0.188891,0.295868,1099,3598,1633,15450


Best threshold by F1: 0.51
Best threshold by balanced accuracy: 0.47


,threshold_name,split,threshold,alert_rate,accuracy,balanced_accuracy,precision,recall,specificity,false_positive_rate,f1,true_positives,false_positives,false_negatives,true_negatives
0,best_f1,valid,0.51,0.246235,0.739440,0.612435,0.225620,0.442899,0.781972,0.218028,0.298950,1210,4153,1522,14895
1,best_f1,test,0.51,0.279508,0.722857,0.630092,0.204520,0.510561,0.749623,0.250377,0.292051,1837,7145,1761,21392
2,best_balanced_accuracy,valid,0.47,0.382094,0.641139,0.620353,0.194545,0.592606,0.648100,0.351900,0.292926,1619,6703,1113,12345
3,best_balanced_accuracy,test,0.47,0.377532,0.648794,0.635148,0.183152,0.617565,0.652732,0.347268,0.282517,2222,9910,1376,18627
4,default_0_50,valid,0.50,0.278421,0.716713,0.615589,0.216524,0.480600,0.750577,0.249423,0.298545,1313,4751,1419,14297
5,default_0_50,test,0.50,0.303781,0.704310,0.630823,0.197603,0.536131,0.725514,0.274486,0.288772,1929,7833,1669,20704



Test confusion matrix at best F1 threshold:


,pred_0_no_big_downside,pred_1_big_downside
actual_0_no_big_downside,21392,7145
actual_1_big_downside,1761,1837


## Downside Risk Threshold Analysis Summary

The downside-risk model predicts:

`target_big_downside_10pct_20d`

This target means:

- `1`: The stock fell at least 10% at any point in the next 20 trading days.
- `0`: The stock did not fall 10% in the next 20 trading days.

This target is imbalanced:

| Split | Big downside rate |
|---|---:|
| Train | 14.76% |
| Validation | 12.54% |
| Test | 11.20% |

Because most rows are `0_no_big_downside`, accuracy alone is misleading. A dummy model can achieve high accuracy by predicting no downside for every row, but it catches zero actual downside events.

---

## Baseline Model Comparison

| Model | Test ROC AUC | Test Average Precision | Test Precision | Test Recall |
|---|---:|---:|---:|---:|
| Dummy | 0.5000 | 0.1120 | 0.00% | 0.00% |
| Logistic Regression Balanced | 0.5521 | 0.1265 | 13.01% | 38.05% |
| Random Forest Downside Balanced | 0.6894 | 0.2107 | 19.76% | 53.61% |

The Random Forest downside model clearly performs better than both dummy and logistic regression.

The test base downside rate is only 11.20%, while the Random Forest average precision is 21.07%. This means the model is meaningfully better than random selection at identifying high-risk stock-date rows.

---

## Why Threshold Tuning Is Needed

The Random Forest model outputs a probability of big downside risk. A threshold converts that probability into a final alert.

Example logic:

    If predicted downside probability >= threshold:
        alert = high downside risk
    else:
        alert = not high downside risk

Different thresholds create different trade-offs:

- Lower threshold: catches more downside events, but creates more false alerts.
- Higher threshold: creates fewer alerts, but misses more actual downside events.

For a risk-warning system like MarketGuard, recall is important because missing dangerous cases is costly. However, precision also matters because too many false alerts reduce trust.

---

## Threshold Comparison on Test Data

| Threshold | Purpose | Alert Rate | Precision | Recall | Balanced Accuracy |
|---:|---|---:|---:|---:|---:|
| 0.51 | Best F1 threshold | 27.95% | 20.45% | 51.06% | 63.01% |
| 0.47 | Best balanced accuracy threshold | 37.75% | 18.32% | 61.76% | 63.51% |
| 0.50 | Default threshold | 30.38% | 19.76% | 53.61% | 63.08% |

---

## Threshold 0.51: Selective High-Confidence Alert

At threshold `0.51`, the model is more selective.

Test confusion matrix:

|  | Predicted no big downside | Predicted big downside |
|---|---:|---:|
| Actual no big downside | 21,392 | 7,145 |
| Actual big downside | 1,761 | 1,837 |

Interpretation:

- The model caught 1,837 out of 3,598 actual big downside cases.
- Recall was 51.06%.
- Precision was 20.45%.
- Alert rate was 27.95%.

This threshold is useful when we want fewer, higher-confidence risk alerts.

---

## Threshold 0.47: Defensive Risk Alert

At threshold `0.47`, the model catches more downside events.

| Metric | Value |
|---|---:|
| True positives | 2,222 |
| False positives | 9,910 |
| False negatives | 1,376 |
| True negatives | 18,627 |
| Recall | 61.76% |
| Precision | 18.32% |
| Alert rate | 37.75% |
| Balanced accuracy | 63.51% |

Interpretation:

- The model catches more actual downside cases than the 0.51 threshold.
- Recall improves from 51.06% to 61.76%.
- Precision drops from 20.45% to 18.32%.
- Alert rate increases from 27.95% to 37.75%.

This threshold is useful when the system should be more defensive and catch more possible danger, even if it creates more false alerts.

---

## Threshold 0.50: Middle Ground

At threshold `0.50`, the model gives a balanced middle option.

| Metric | Value |
|---|---:|
| Alert rate | 30.38% |
| Precision | 19.76% |
| Recall | 53.61% |
| Balanced accuracy | 63.08% |
| True positives | 1,929 |
| False positives | 7,833 |
| False negatives | 1,669 |
| True negatives | 20,704 |

This threshold is a reasonable simple default. It catches more downside events than the 0.51 threshold, while producing fewer false alerts than the 0.47 threshold.

---

## Final Interpretation

The downside Random Forest model is useful for identifying stocks with elevated 20-day downside risk.

The best threshold depends on how the model will be used:

| Threshold | Best Use Case |
|---:|---|
| 0.51 | Selective high-confidence risk alerts |
| 0.47 | Defensive risk monitoring |
| 0.50 | Simple middle-ground default |

For MarketGuard India, the better approach is to use the predicted downside probability as a risk score and convert it into risk bands.

Example risk bands:

| Risk Band | Probability Range |
|---|---|
| Low Risk | probability < 0.40 |
| Watch Risk | 0.40 to 0.47 |
| High Risk | 0.47 to 0.51 |
| Very High Risk | probability >= 0.51 |

This is better than using only a binary `alert / no alert` decision because it preserves more information from the model.

The final dashboard should show the downside probability and risk band together, instead of only showing a hard prediction.

### Downside probability decile analysis

Now we should do downside probability decile analysis. This will tell us whether stocks with high predicted downside probability actually had much higher downside rates.

In [37]:
DOWNSIDE_DECILE_REPORT_PATH = REPORT_DIR / "downside_10pct_rf_decile_analysis_v1.csv"

def downside_probability_decile_analysis(
    eval_df: pd.DataFrame,
    proba: np.ndarray,
    split_name: str,
) -> pd.DataFrame:
    """
    Check whether higher predicted downside probability groups
    actually have higher downside risk.

    Decile 10 = highest predicted downside probability.
    Decile 1 = lowest predicted downside probability.
    """
    temp = eval_df.copy()
    temp["predicted_downside_probability"] = proba

    temp["probability_rank"] = temp["predicted_downside_probability"].rank(method="first")

    temp["probability_decile"] = pd.qcut(
        temp["probability_rank"],
        q=10,
        labels=False,
    ) + 1

    summary = (
        temp
        .groupby("probability_decile")
        .agg(
            rows=("target_big_downside_10pct_20d", "size"),
            avg_predicted_downside_probability=("predicted_downside_probability", "mean"),
            actual_big_downside_rate=("target_big_downside_10pct_20d", "mean"),
            actual_5pct_downside_rate=("target_downside_5pct_20d", "mean"),
            actual_outperform_rate=("target_outperform_nifty50_20d", "mean"),
            avg_future_return_20d=("future_return_20d", "mean"),
            avg_future_min_return_20d=("future_min_return_20d", "mean"),
            avg_future_max_return_20d=("future_max_return_20d", "mean"),
            avg_excess_return_vs_nifty50=("future_excess_return_20d_vs_nifty50", "mean"),
        )
        .reset_index()
        .sort_values("probability_decile", ascending=False)
    )

    summary["split"] = split_name

    return summary


valid_downside_eval_df = model_data.loc[
    valid_mask,
    [
        "date",
        "symbol",
        "yf_ticker",
        "future_return_20d",
        "future_min_return_20d",
        "future_max_return_20d",
        "future_excess_return_20d_vs_nifty50",
        "target_outperform_nifty50_20d",
        "target_downside_5pct_20d",
        "target_big_downside_10pct_20d",
    ],
].copy()

test_downside_eval_df = model_data.loc[
    test_mask,
    [
        "date",
        "symbol",
        "yf_ticker",
        "future_return_20d",
        "future_min_return_20d",
        "future_max_return_20d",
        "future_excess_return_20d_vs_nifty50",
        "target_outperform_nifty50_20d",
        "target_downside_5pct_20d",
        "target_big_downside_10pct_20d",
    ],
].copy()


downside_valid_decile_summary = downside_probability_decile_analysis(
    eval_df=valid_downside_eval_df,
    proba=downside_rf_valid_proba,
    split_name="valid",
)

downside_test_decile_summary = downside_probability_decile_analysis(
    eval_df=test_downside_eval_df,
    proba=downside_rf_test_proba,
    split_name="test",
)

downside_decile_summary = pd.concat(
    [downside_valid_decile_summary, downside_test_decile_summary],
    ignore_index=True,
)

print("Downside RF validation decile summary:")
display(downside_valid_decile_summary)

print("\nDownside RF test decile summary:")
display(downside_test_decile_summary)

print("\nTop risk decile vs bottom risk decile:")

downside_top_bottom_summary = downside_decile_summary[
    downside_decile_summary["probability_decile"].isin([1, 10])
].copy()

display(downside_top_bottom_summary)

downside_decile_summary.to_csv(DOWNSIDE_DECILE_REPORT_PATH, index=False)

print("\nSaved:")
print(DOWNSIDE_DECILE_REPORT_PATH)

Downside RF validation decile summary:


,probability_decile,rows,avg_predicted_downside_probability,actual_big_downside_rate,actual_5pct_downside_rate,actual_outperform_rate,avg_future_return_20d,avg_future_min_return_20d,avg_future_max_return_20d,avg_excess_return_vs_nifty50,split
9,10,2178,0.605653,0.277778,0.606979,0.428375,-0.012759,-0.074417,0.060317,-0.007319,valid
8,9,2178,0.544901,0.197888,0.511478,0.448577,-0.000447,-0.057245,0.057210,-0.002120,valid
7,8,2178,0.509008,0.156566,0.428834,0.49449,0.014921,-0.048153,0.067677,0.005864,valid
6,7,2178,0.479142,0.130854,0.411387,0.511478,0.018973,-0.046139,0.068531,0.009498,valid
5,6,2178,0.451258,0.108356,0.389807,0.500918,0.020871,-0.044122,0.066080,0.009432,valid
4,5,2178,0.422901,0.097796,0.337466,0.523416,0.024902,-0.039242,0.065847,0.011873,valid
3,4,2178,0.390325,0.082185,0.312672,0.527548,0.024598,-0.037877,0.063067,0.013032,valid
2,3,2178,0.355928,0.061065,0.282369,0.541322,0.023940,-0.034474,0.059333,0.012926,valid
1,2,2178,0.318272,0.060606,0.278696,0.557392,0.021110,-0.035038,0.055208,0.013874,valid
0,1,2178,0.260375,0.081267,0.344353,0.511938,0.007046,-0.040794,0.048556,0.007722,valid



Downside RF test decile summary:


,probability_decile,rows,avg_predicted_downside_probability,actual_big_downside_rate,actual_5pct_downside_rate,actual_outperform_rate,avg_future_return_20d,avg_future_min_return_20d,avg_future_max_return_20d,avg_excess_return_vs_nifty50,split
9,10,3214,0.643737,0.242999,0.545426,0.548538,0.018667,-0.060425,0.072055,0.015865,test
8,9,3213,0.569257,0.190165,0.483038,0.547463,0.014369,-0.057756,0.066273,0.013646,test
7,8,3214,0.522838,0.163659,0.443684,0.503734,0.006208,-0.051772,0.056443,0.005203,test
6,7,3213,0.481141,0.119203,0.381886,0.505447,0.007707,-0.045093,0.054275,0.005953,test
5,6,3213,0.441331,0.120759,0.386866,0.485216,0.004277,-0.045063,0.052003,0.003864,test
4,5,3214,0.392183,0.095208,0.310516,0.521157,0.008507,-0.039484,0.049656,0.004114,test
3,4,3213,0.330393,0.068161,0.272331,0.538438,0.012707,-0.036659,0.050515,0.006711,test
2,3,3214,0.258607,0.046982,0.241755,0.541693,0.015640,-0.032239,0.048448,0.011515,test
1,2,3213,0.188909,0.037037,0.255524,0.495798,0.008777,-0.033121,0.040863,0.005329,test
0,1,3214,0.120876,0.03547,0.204107,0.485376,0.005180,-0.030769,0.036601,0.002668,test



Top risk decile vs bottom risk decile:


,probability_decile,rows,avg_predicted_downside_probability,actual_big_downside_rate,actual_5pct_downside_rate,actual_outperform_rate,avg_future_return_20d,avg_future_min_return_20d,avg_future_max_return_20d,avg_excess_return_vs_nifty50,split
0,10,2178,0.605653,0.277778,0.606979,0.428375,-0.012759,-0.074417,0.060317,-0.007319,valid
9,1,2178,0.260375,0.081267,0.344353,0.511938,0.007046,-0.040794,0.048556,0.007722,valid
10,10,3214,0.643737,0.242999,0.545426,0.548538,0.018667,-0.060425,0.072055,0.015865,test
19,1,3214,0.120876,0.03547,0.204107,0.485376,0.005180,-0.030769,0.036601,0.002668,test



Saved:
E:\Projects\marketguard-india\reports\baseline_modeling\downside_10pct_rf_decile_analysis_v1.csv


## Downside Decile Analysis: Simple Explanation

This table checks whether the downside-risk model is ranking stocks correctly.

The model predicts:

`target_big_downside_10pct_20d`

This means:

- `1`: The stock fell at least 10% at some point in the next 20 trading days.
- `0`: The stock did not fall 10% in the next 20 trading days.

---

## What is a decile?

The model gives each stock-date row a downside-risk probability.

Then we sort all rows by that probability and divide them into 10 equal groups.

| Decile | Meaning |
|---:|---|
| Decile 10 | Stocks with the highest predicted downside risk |
| Decile 1 | Stocks with the lowest predicted downside risk |

So the main comparison is:

`Decile 10 vs Decile 1`

If the model is useful, Decile 10 should have more actual downside events than Decile 1.

---

## Main columns to look at

### 1. `avg_predicted_downside_probability`

This is the model's average predicted risk score for each decile.

Example from test data:

| Decile | Avg predicted downside probability |
|---:|---:|
| 10 | 64.37% |
| 1 | 12.09% |

This means the model thinks Decile 10 is much riskier than Decile 1.

---

### 2. `actual_big_downside_rate`

This is the most important column.

It shows how many stocks actually fell at least 10% in the next 20 trading days.

Example from test data:

| Decile | Actual 10% downside rate |
|---:|---:|
| 10 | 24.30% |
| 1 | 3.55% |

This means the highest-risk group had much more actual downside risk than the lowest-risk group.

So the model is ranking risk correctly.

---

### 3. `actual_5pct_downside_rate`

This shows how many stocks fell at least 5% in the next 20 trading days.

Example from test data:

| Decile | Actual 5% downside rate |
|---:|---:|
| 10 | 54.54% |
| 1 | 20.41% |

This shows that the model is also good at identifying moderate downside risk.

---

### 4. `avg_future_min_return_20d`

This shows the average worst fall during the next 20 trading days.

Example from test data:

| Decile | Avg future minimum return |
|---:|---:|
| 10 | -6.04% |
| 1 | -3.08% |

This means Decile 10 stocks had deeper drawdowns than Decile 1 stocks.

This is very important for MarketGuard because we care about how much the stock can fall during the holding period.

---

## Important note

`avg_future_return_20d` is not the main column for this model.

This model is not predicting whether the stock will end negative after 20 days.

It is predicting whether the stock will fall sharply at any point during the next 20 days.

A stock can fall 10% during the period and still recover by the end of 20 days.

That is why `avg_future_return_20d` can still be positive even in the high-risk decile.

---

## Main test result

| Metric | Decile 10 | Decile 1 |
|---|---:|---:|
| Avg predicted downside probability | 64.37% | 12.09% |
| Actual 10% downside rate | 24.30% | 3.55% |
| Actual 5% downside rate | 54.54% | 20.41% |
| Avg future minimum return | -6.04% | -3.08% |

---

## Final interpretation

The downside-risk model is useful.

The stocks with higher predicted downside probability actually had:

- More 10% downside events.
- More 5% downside events.
- Deeper future drawdowns.

So this model is good for ranking stocks by downside risk.

For MarketGuard India, this is more useful than a simple buy/sell prediction because it helps identify which stocks may be risky over the next 20 trading days.

### Create final downside-risk report only

In [38]:
from datetime import datetime

DOWNSIDE_RF_MARKDOWN_REPORT_PATH = (
    REPORT_DIR / "downside_10pct_rf_final_summary_report_v1.md"
)

markdown_report = f"""
# Downside Risk Model Final Summary Report

Generated on: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## Target

The model predicts:

`target_big_downside_10pct_20d`

This target means:

- `1`: The stock fell at least 10% at any point in the next 20 trading days.
- `0`: The stock did not fall 10% in the next 20 trading days.

This is a path-risk target. It checks whether the stock had a large drawdown at any point during the next 20 trading days, not only whether the final 20-day return was negative.

---

## Target Distribution

| Split | Big Downside Rate |
|---|---:|
| Train | 14.76% |
| Validation | 12.54% |
| Test | 11.20% |

The target is imbalanced because most stock-date rows do not experience a 10% drawdown in the next 20 trading days.

Because of this, accuracy alone is misleading. A dummy model can get high accuracy by always predicting no big downside, but it catches zero actual downside events.

---

## Baseline Model Comparison

| Model | Test ROC AUC | Test Average Precision | Test Precision | Test Recall |
|---|---:|---:|---:|---:|
| Dummy | 0.5000 | 0.1120 | 0.00% | 0.00% |
| Logistic Regression Balanced | 0.5521 | 0.1265 | 13.01% | 38.05% |
| Random Forest Downside Balanced | 0.6894 | 0.2107 | 19.76% | 53.61% |

The Random Forest downside model clearly performs better than both the dummy baseline and logistic regression.

The test base downside rate is only 11.20%, while the Random Forest average precision is 21.07%. This means the model is meaningfully better than random selection at identifying high-risk stock-date rows.

---

## Final Selected Model

The selected downside-risk model is:

`random_forest_downside_balanced`

Model type:

`RandomForestClassifier`

The model uses class balancing:

`class_weight="balanced_subsample"`

This helps the model pay more attention to the minority class, which is the actual big downside event.

---

## Final Random Forest Performance

| Split | ROC AUC | Average Precision | Precision | Recall | F1 | Balanced Accuracy |
|---|---:|---:|---:|---:|---:|---:|
| Train | 0.8050 | 0.4490 | 30.11% | 76.37% | 0.4319 | 0.7284 |
| Validation | 0.6554 | 0.2227 | 21.65% | 48.06% | 0.2985 | 0.6156 |
| Test | 0.6894 | 0.2107 | 19.76% | 53.61% | 0.2888 | 0.6308 |

The test ROC AUC is `0.6894`, which is much better than the random baseline of `0.5000`.

This shows that the model has useful signal for downside-risk prediction.

---

## Threshold Analysis

The model outputs a downside-risk probability. A threshold converts that probability into a binary risk alert.

Example logic:

    If predicted downside probability >= threshold:
        alert = high downside risk
    else:
        alert = not high downside risk

Different thresholds create different trade-offs:

- Lower threshold: catches more downside events, but creates more false alerts.
- Higher threshold: creates fewer alerts, but misses more actual downside events.

For a risk-warning system like MarketGuard, recall is important because missing dangerous cases is costly. However, precision also matters because too many false alerts reduce trust.

### Threshold Comparison on Test Data

| Threshold | Purpose | Alert Rate | Precision | Recall | Balanced Accuracy |
|---:|---|---:|---:|---:|---:|
| 0.51 | Selective high-confidence alert | 27.95% | 20.45% | 51.06% | 63.01% |
| 0.47 | Defensive risk monitoring | 37.75% | 18.32% | 61.76% | 63.51% |
| 0.50 | Middle-ground default | 30.38% | 19.76% | 53.61% | 63.08% |

### Threshold Interpretation

| Threshold | Best Use Case |
|---:|---|
| 0.51 | Selective high-confidence alerts |
| 0.47 | Defensive risk monitoring |
| 0.50 | Simple middle-ground default |

For MarketGuard India, the better approach is to use the predicted downside probability as a risk score instead of only using one hard threshold.

---

## Decile Risk Ranking Analysis

The model's probability scores were divided into 10 deciles.

| Decile | Meaning |
|---:|---|
| Decile 10 | Stocks with the highest predicted downside risk |
| Decile 1 | Stocks with the lowest predicted downside risk |

A useful risk model should show higher actual downside rates in Decile 10 than Decile 1.

---

## Validation: Top Risk Decile vs Bottom Risk Decile

| Metric | Decile 10 | Decile 1 |
|---|---:|---:|
| Avg predicted downside probability | 60.57% | 26.04% |
| Actual 10% downside rate | 27.78% | 8.13% |
| Actual 5% downside rate | 60.70% | 34.44% |
| Avg future minimum return | -7.44% | -4.08% |
| Avg future 20-day return | -1.28% | +0.70% |
| Avg excess return vs Nifty 50 | -0.73% | +0.77% |

Validation big downside lift:

`3.42x`

This means the highest-risk validation decile had around 3.4 times the actual 10% downside rate of the lowest-risk decile.

---

## Test: Top Risk Decile vs Bottom Risk Decile

| Metric | Decile 10 | Decile 1 |
|---|---:|---:|
| Avg predicted downside probability | 64.37% | 12.09% |
| Actual 10% downside rate | 24.30% | 3.55% |
| Actual 5% downside rate | 54.54% | 20.41% |
| Avg future minimum return | -6.04% | -3.08% |
| Avg future 20-day return | +1.87% | +0.52% |
| Avg excess return vs Nifty 50 | +1.59% | +0.27% |

Test big downside lift:

`6.85x`

This means the highest-risk test decile had around 6.9 times the actual 10% downside rate of the lowest-risk decile.

This is strong evidence that the model ranks downside risk correctly.

---

## Important Interpretation

In the test split, the high-risk decile still had a positive average 20-day return.

This is not a contradiction.

The model is not predicting:

Will the stock end negative after 20 days?

The model is predicting:

Will the stock fall at least 10% at any point during the next 20 trading days?

A stock can fall sharply during the 20-day window and still recover by the end of the period.

This is why the downside model is useful for path-risk monitoring.

---

## Recommended Risk Bands

Instead of only using a binary alert, MarketGuard should use risk bands.

| Risk Band | Probability Range |
|---|---|
| Low Risk | probability < 0.40 |
| Watch Risk | 0.40 to 0.47 |
| High Risk | 0.47 to 0.51 |
| Very High Risk | probability >= 0.51 |

These risk bands preserve more information than a simple alert/no-alert output.

---

## Final Conclusion

The downside Random Forest model is useful for MarketGuard India.

The model successfully ranks stocks by future downside risk.

Higher predicted downside probability is associated with:

- Higher actual 10% downside rate.
- Higher actual 5% downside rate.
- Deeper future drawdowns.

The downside model is stronger and more directly useful than the outperform model because MarketGuard's main purpose is risk monitoring, not simple buy/sell prediction.

This model should be used as the first baseline downside-risk model for the dashboard.
"""

DOWNSIDE_RF_MARKDOWN_REPORT_PATH.write_text(markdown_report, encoding="utf-8")

print("Saved final downside-risk markdown report:")
print(DOWNSIDE_RF_MARKDOWN_REPORT_PATH)

Saved final downside-risk markdown report:
E:\Projects\marketguard-india\reports\baseline_modeling\downside_10pct_rf_final_summary_report_v1.md


### Save downside model + feature metadata

In [39]:
import json
import joblib
from datetime import datetime

DOWNSIDE_RF_MODEL_PATH = MODEL_DIR / "random_forest_downside_10pct_20d_v1.joblib"
DOWNSIDE_FEATURE_LIST_PATH = REPORT_DIR / "downside_10pct_feature_list_v1.csv"
DOWNSIDE_MODEL_METADATA_PATH = REPORT_DIR / "downside_10pct_model_metadata_v1.json"

# Save trained downside model
joblib.dump(downside_rf_model, DOWNSIDE_RF_MODEL_PATH)

# Save feature list
downside_feature_list_df = pd.DataFrame({
    "feature_order": range(1, len(feature_cols) + 1),
    "feature": feature_cols,
})

downside_feature_list_df.to_csv(DOWNSIDE_FEATURE_LIST_PATH, index=False)

# Save model metadata
downside_model_metadata = {
    "model_name": "random_forest_downside_balanced",
    "model_type": "RandomForestClassifier",
    "target": "target_big_downside_10pct_20d",
    "target_meaning": "1 means the stock fell at least 10% at any point in the next 20 trading days.",
    "feature_count": len(feature_cols),
    "train_rows": int(len(X_train)),
    "valid_rows": int(len(X_valid)),
    "test_rows": int(len(X_test)),
    "train_positive_rate": float(y_train_downside.mean()),
    "valid_positive_rate": float(y_valid_downside.mean()),
    "test_positive_rate": float(y_test_downside.mean()),
    "selected_thresholds": {
        "selective_high_confidence_alert": 0.51,
        "defensive_risk_monitoring": 0.47,
        "middle_ground_default": 0.50,
    },
    "recommended_risk_bands": {
        "low_risk": "probability < 0.40",
        "watch_risk": "0.40 <= probability < 0.47",
        "high_risk": "0.47 <= probability < 0.51",
        "very_high_risk": "probability >= 0.51",
    },
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

with open(DOWNSIDE_MODEL_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(downside_model_metadata, f, indent=4)

print("Saved downside model:")
print(DOWNSIDE_RF_MODEL_PATH)

print("\nSaved feature list:")
print(DOWNSIDE_FEATURE_LIST_PATH)

print("\nSaved model metadata:")
print(DOWNSIDE_MODEL_METADATA_PATH)

print("\nFeature count:", len(feature_cols))
print("Target:", downside_model_metadata["target"])

Saved downside model:
E:\Projects\marketguard-india\models\random_forest_downside_10pct_20d_v1.joblib

Saved feature list:
E:\Projects\marketguard-india\reports\baseline_modeling\downside_10pct_feature_list_v1.csv

Saved model metadata:
E:\Projects\marketguard-india\reports\baseline_modeling\downside_10pct_model_metadata_v1.json

Feature count: 79
Target: target_big_downside_10pct_20d
